# Маппинг hxid-биоматериал-тип пробирки

Код разбит на следующие блоки:

1. Выгрузка данных из базы `phr`
2. Очистка данных по `effectiveDateTime` внутри `body`
3. Загрузка `visits.csv` из Metabase
4. Агрегация заказов в корзины по `visit_id`
5. Скачать `patient_ids.csv` из Metabase (mapping `visit_id` → `patient_id`)
6. Подмешивание `patient_id` к агрегированным визитам
7. Парсинг `body` в `hxid` и соответствующие `Observation`

Сырая выгрузка из PHR на шаге 1 сохраняется в `bundle_v2.csv`; на шаге 2 этот файл **один раз** загружается в память. Дальше промежуточные таблицы не пишутся на диск; в конце пайплайна сохраняются только JSON.

## Важное примечание

В рамках задачи DBA-202 команда DBA выгружала деперсонализированные данные из базы `phr`.

При выгрузке данных они ориентировались на колонку `moment_utc`. Проблема в том, что эта дата отражает дату добавления или обновления записи в БД, а не реальную дату заказа.

Реальная дата заказа находится в `effectiveDateTime` внутри `bundle`, который лежит в колонке `body`.

Это означает, что после первичной выгрузки данные нужно дополнительно очищать и оставлять только нужный месяц или нужный год по `effectiveDate`.

In [1]:
!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.2.3

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
  Using cached numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (89 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl (20.6 MB)
Using cached pandas-2.2.3-cp311-cp311-macosx_10_9_x86_64.whl (12.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
label-studio-sdk 2.0.19 requires numpy<3.0.0,>=2.2, but you have numpy 1.26.4 which is incompatible.
label-studio-sdk 2.0.19 requires urllib3>=2.5.0, but you have urllib3 1.26.20 which is incompatible.
opencv-python-headless 4.13.0.92

In [3]:
import numpy as np
import pandas as pd

print(np.__version__)
print(pd.__version__)

1.26.4
2.2.3


In [4]:
from __future__ import annotations

import ast
import csv
import json
import requests
import sys
from collections import defaultdict
from datetime import date, datetime
from pathlib import Path

import pandas as pd
import psycopg2

csv.field_size_limit(sys.maxsize)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

DB_HOST = ""
DB_NAME = "phr_r4_year"
DB_USER = "postgres"
DB_PASSWORD = ""

BUNDLE_TABLE = "bundle_v2"
BUNDLE_CSV = "bundle_v2.csv"  # шаг 1: выгрузка из БД; шаг 2: загрузка с диска
VISITS_CSV = "visits.csv"
PATIENT_IDS_CSV = "patient_ids.csv"
VISIT_HXID_JSON = "visit_hxid_observations.json"
VISIT_HXID_WITH_PATIENTS_JSON = "visit_hxid_observations_with_patient_ids.json"

START_DATE = date(2025, 1, 1)
END_DATE = date(2026, 3, 31)

## 1. Выгрузка данных за март или за год из дампа БД `phr`

Скрипт выгрузки данных за год или за март из дампа БД `phr`.

Примечание: креды указываются прямо в ноутбуке. Для месячной выгрузки используется база `phr_r4_month`, для годовой выгрузки - `phr_r4_year`.

In [16]:
# Шаг 1. Выгрузка таблицы bundle_v2 из PHR и сохранение в CSV.

conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)

query = f"SELECT * FROM {BUNDLE_TABLE}"
bundle_df = pd.read_sql(query, conn)
bundle_df.to_csv(BUNDLE_CSV, index=False)
conn.close()

print(f"Сохранено строк: {len(bundle_df):,}".replace(",", " "))
print(f"Файл: {BUNDLE_CSV}")
print(bundle_df.head(3))

/var/folders/1d/9x32vtfd74154hj94nzb6kxc0000gq/T/ipykernel_21912/1201167830.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bundle_df = pd.read_sql(query, conn)


Сохранено строк: 225 139
Файл: bundle_v2.csv
                                   o_id  v_id                 moment_utc  \
0  b0d6cd84-8303-42a0-80f2-5be6fff27be3     3 2025-04-03 07:23:07.281040   
1  3f9b1024-91ba-4835-9ed1-3b1b76d2f36b     3 2025-04-03 07:23:07.858025   
2  75f9dc7b-2b51-4210-827a-fef1df067617     1 2025-04-03 07:24:23.453731   

   is_deleted  \
0       False   
1       False   
2       False   

                                                                                                                                                                                                      body  
0  {'meta': {'tags': {}, 'profiles': [], 'permissions': {'Read': ['los_client', 'medzoom', 'service'], 'Owner': ['los_client'], 'UpdateBody': ['los_client'], 'ReadHistory': ['los_client']}, 'security...  
1  {'meta': {'tags': {}, 'profiles': [], 'permissions': {'Read': ['los_client', 'medzoom', 'service'], 'Owner': ['los_client'], 'UpdateBody': ['los_client'], 'ReadHistory'

## 2. Очистка данных: оставляем только данные за год

Здесь читаем `bundle_v2.csv` с диска; дальше по пайплайну данные идут только в памяти, пока в конце не сохранятся JSON.

Аналогично можно оставить данные за март.

P.S. Этот блок работает долго, так как данных очень много.

Логика фильтрации:

- ориентируемся не на `moment_utc`, а на `effectiveDateTime` внутри `body`
- если в `body` несколько `DiagnosticReport`, оставляем только те, что попадают в нужный диапазон дат
- связанные `Observation` тоже очищаем
- если дата у `DiagnosticReport` не найдена, используем `Composition.date` как fallback

In [5]:
def parse_iso_date(value: str):
    value = (value or "").strip()
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).date()
    except ValueError:
        return None


def get_composition_date(doc: dict):
    for entry in doc.get("entry", []) or []:
        resource = entry.get("resource", {})
        if resource.get("resourceType") == "Composition":
            dt = parse_iso_date(resource.get("date"))
            if dt:
                return dt
    return None


def get_resource_reference_keys(entry: dict, resource: dict):
    keys = set()
    full_url = (entry.get("fullUrl") or "").strip()
    resource_id = (resource.get("id") or "").strip()
    resource_type = (resource.get("resourceType") or "").strip()

    if full_url:
        keys.add(full_url)
    if resource_id:
        keys.add(resource_id)
        if resource_type:
            keys.add(f"{resource_type}/{resource_id}")
    return keys


def trim_body_to_date_range(body_text: str, start_date: date, end_date: date):
    doc = ast.literal_eval(body_text)
    entries = doc.get("entry", []) or []
    composition_date = get_composition_date(doc)
    order_id = ((doc.get("identifier") or {}).get("value") or "").strip()

    kept_report_refs = set()
    kept_observation_refs = set()
    removed_reports = 0
    removed_observations = 0

    for entry in entries:
        resource = entry.get("resource", {})
        if resource.get("resourceType") != "DiagnosticReport":
            continue

        report_date = parse_iso_date(resource.get("effectiveDateTime")) or composition_date
        if report_date is None:
            removed_reports += 1
            continue
        if not (start_date <= report_date <= end_date):
            removed_reports += 1
            continue

        kept_report_refs.update(get_resource_reference_keys(entry, resource))
        for ref in resource.get("result", []) or []:
            reference = (ref.get("reference") or "").strip()
            if reference:
                kept_observation_refs.add(reference)

    if not kept_report_refs:
        return None, removed_reports, removed_observations, order_id

    new_entries = []
    for entry in entries:
        resource = entry.get("resource", {})
        resource_type = resource.get("resourceType")

        if resource_type == "DiagnosticReport":
            if get_resource_reference_keys(entry, resource) & kept_report_refs:
                new_entries.append(entry)
            continue

        if resource_type == "Observation":
            obs_refs = get_resource_reference_keys(entry, resource)
            obs_date = parse_iso_date(resource.get("effectiveDateTime")) or composition_date
            if (obs_refs & kept_observation_refs) and (
                obs_date is None or start_date <= obs_date <= end_date
            ):
                new_entries.append(entry)
            else:
                removed_observations += 1
            continue

        if resource_type == "Composition":
            sections = resource.get("section") or []
            for section in sections:
                section_entries = section.get("entry") or []
                section["entry"] = [
                    item
                    for item in section_entries
                    if (item.get("reference") or "").strip() in kept_report_refs
                ]
            new_entries.append(entry)
            continue

        new_entries.append(entry)

    doc["entry"] = new_entries
    return repr(doc), removed_reports, removed_observations, order_id

In [ ]:
# Шаг 2. Загрузка `bundle_v2.csv` и очистка по effectiveDateTime внутри body.

print("Start loading data...")

bundle_df = pd.read_csv(BUNDLE_CSV, dtype=str, low_memory=False)
bundle_df = bundle_df.fillna("")

kept_rows = []
skipped_parse_errors = 0
rows_without_body = 0
rows_dropped_after_trim = 0
total_removed_reports = 0
total_removed_observations = 0
changed_rows = 0
log_every = 1000

print(
    f"Start filtering: rows={len(bundle_df):,} | period={START_DATE}..{END_DATE}".replace(",", " ")
)

for i, row in enumerate(bundle_df.to_dict("records"), start=1):
    body_text = row.get("body", "")
    if not body_text.strip():
        rows_without_body += 1
        continue

    try:
        trimmed_body, removed_reports, removed_observations, order_id = trim_body_to_date_range(
            body_text,
            START_DATE,
            END_DATE,
        )
    except Exception:
        skipped_parse_errors += 1
        continue

    total_removed_reports += removed_reports
    total_removed_observations += removed_observations

    if removed_reports > 0 or removed_observations > 0:
        changed_rows += 1

    if i % log_every == 0:
        print(
            (
                f"Scanned rows: {i:,} | kept rows: {len(kept_rows):,} | "
                f"changed rows: {changed_rows:,} | removed reports: {total_removed_reports:,} | "
                f"removed observations: {total_removed_observations:,} | parse errors: {skipped_parse_errors:,}"
            ).replace(",", " ")
        )

    if trimmed_body:
        row["body"] = trimmed_body
        kept_rows.append(row)
    else:
        rows_dropped_after_trim += 1

filtered_bundle_df = pd.DataFrame(kept_rows)

print("Filtering finished")
print(f"Finished scanning rows: {len(bundle_df):,}".replace(",", " "))
print(f"Rows without body: {rows_without_body:,}".replace(",", " "))
print(f"Kept rows: {len(filtered_bundle_df):,}".replace(",", " "))
print(f"Dropped rows after trim: {rows_dropped_after_trim:,}".replace(",", " "))
print(f"Changed rows: {changed_rows:,}".replace(",", " "))
print(f"Removed reports total: {total_removed_reports:,}".replace(",", " "))
print(f"Removed observations total: {total_removed_observations:,}".replace(",", " "))
print(f"Skipped parse errors: {skipped_parse_errors:,}".replace(",", " "))
print(filtered_bundle_df.head(3))

Start loading data...
Start filtering: rows=225 139 | period=2025-01-01..2026-03-31
Scanned rows: 1 000 | kept rows: 993 | changed rows: 49 | removed reports: 77 | removed observations: 369 | parse errors: 0
Scanned rows: 2 000 | kept rows: 1 988 | changed rows: 91 | removed reports: 161 | removed observations: 709 | parse errors: 0
Scanned rows: 3 000 | kept rows: 2 986 | changed rows: 116 | removed reports: 199 | removed observations: 947 | parse errors: 0


## 3. Скачать список `visit_id` с соответствующими `lab_id` из Metabase

Для этого:

- зайти в Metabase в отчёт `Lab Order`
- скрыть все столбцы кроме `Super Task Id` и `Visit Id`
- установить фильтр по датам, например за март
- скачать CSV-файл и переименовать его в `visits.csv`

## 4. Сгруппировать все заказы по `visit_id`

В выгрузке `bundle_v2` каждая строка — отдельный заказ, но внутри одной корзины может быть несколько заказов.

Поэтому здесь мы:

- берём `super_task_id` из `body.identifier.value`
- маппим его в `visit_id` через `visits.csv`
- объединяем все строки, относящиеся к одному `visit_id`, в одну агрегированную запись

In [36]:
# Шаг 4. Агрегация по visit_id: нужен `filtered_bundle_df` из шага 2 и файл `visits.csv` (шаг 3).

def extract_super_task_id(body_text: str):
    doc = ast.literal_eval(body_text)
    return ((doc.get("identifier") or {}).get("value") or "").strip()


visits_df = pd.read_csv(VISITS_CSV, dtype=str).fillna("")
visit_map = dict(
    zip(
        visits_df["Super Task ID"].str.strip(),
        visits_df["Visit ID"].str.strip(),
    )
)

aggregated = defaultdict(list)
visit_super_tasks = defaultdict(set)
matched = 0
unmatched = 0

print("Start agregating...")

for i, row in enumerate(filtered_bundle_df.to_dict("records"), start=1):
    body_text = row.get("body", "")
    if not body_text.strip():
        unmatched += 1
        continue

    try:
        super_task_id = extract_super_task_id(body_text)
    except Exception:
        unmatched += 1
        continue

    visit_id = visit_map.get(super_task_id)
    if not visit_id:
        unmatched += 1
        continue

    matched += 1
    aggregated[visit_id].append({"super_task_id": super_task_id, **row})
    visit_super_tasks[visit_id].add(super_task_id)

    if i % 1000 == 0:
        print(
            f"Scanned rows: {i:,} | matched: {matched:,} | unmatched: {unmatched:,} | visits: {len(aggregated):,}".replace(",", " ")
        )

aggregated_rows = []
for visit_id in sorted(aggregated):
    rows = aggregated[visit_id]
    aggregated_rows.append(
        {
            "visit_id": visit_id,
            "row_count": len(rows),
            "super_task_ids_json": json.dumps(sorted(visit_super_tasks[visit_id]), ensure_ascii=False),
            "rows_json": json.dumps(rows, ensure_ascii=False),
        }
    )

aggregated_visits_df = pd.DataFrame(aggregated_rows)

print(f"Finished | scanned: {len(filtered_bundle_df):,} | matched: {matched:,} | unmatched: {unmatched:,} | output visits: {len(aggregated_visits_df):,}".replace(",", " "))
print(aggregated_visits_df.head(3))

Start agregating...
Scanned rows: 1 000 | matched: 972 | unmatched: 28 | visits: 909
Scanned rows: 2 000 | matched: 1 967 | unmatched: 33 | visits: 1 815
Scanned rows: 3 000 | matched: 2 966 | unmatched: 34 | visits: 2 705
Scanned rows: 4 000 | matched: 3 964 | unmatched: 36 | visits: 3 164
Scanned rows: 5 000 | matched: 4 958 | unmatched: 42 | visits: 4 033
Scanned rows: 6 000 | matched: 5 957 | unmatched: 43 | visits: 4 909
Scanned rows: 7 000 | matched: 6 956 | unmatched: 44 | visits: 5 787
Scanned rows: 8 000 | matched: 7 954 | unmatched: 46 | visits: 6 670
Scanned rows: 9 000 | matched: 8 954 | unmatched: 46 | visits: 7 566
Scanned rows: 10 000 | matched: 9 948 | unmatched: 52 | visits: 8 451
Scanned rows: 11 000 | matched: 10 941 | unmatched: 59 | visits: 9 340
Scanned rows: 12 000 | matched: 11 940 | unmatched: 60 | visits: 10 243
Scanned rows: 13 000 | matched: 12 937 | unmatched: 63 | visits: 11 107
Scanned rows: 14 000 | matched: 13 937 | unmatched: 63 | visits: 11 999
Scanne

## 5. Скачать mapping `visit_id` → `patient_id` из Metabase

- скачайте из Metabase CSV за нужный период с колонками `visit_id` и `patient_id`
- сохраните файл как `patient_ids.csv` в каталоге с ноутбуком

Следующий шаг добавляет эти идентификаторы в таблицу агрегированных визитов.

## 6. Добавление `patient_id` к агрегированным визитам

- берём `aggregated_visits_df`
- по `visit_id` добавляем колонку `patient_id` из `patient_ids.csv`
- таблицу держим в `aggregated_visits_df` в памяти

In [37]:
# Шаг 6. Добавление patient_id к агрегированным визитам


def load_patient_ids(path: str) -> dict[str, str]:
    patient_df = pd.read_csv(path, dtype=str).fillna("")
    visit_col = "Visit ID" if "Visit ID" in patient_df.columns else "visit_id"
    patient_col = "Patient ID" if "Patient ID" in patient_df.columns else "patient_id"
    return dict(zip(patient_df[visit_col].str.strip(), patient_df[patient_col].str.strip()))


patient_ids_by_visit = load_patient_ids(PATIENT_IDS_CSV)
aggregated_visits_df["patient_id"] = (
    aggregated_visits_df["visit_id"].astype(str).str.strip().map(patient_ids_by_visit).fillna("")
)

print(aggregated_visits_df.head(3))

                               visit_id  row_count  \
0  0002eb49-3e1f-4e0b-8480-776aedec342c          1   
1  000962a3-4b82-4ad5-84bc-cda8a39fd7e3          1   
2  000e180b-bad9-4cd4-8425-35a55b8dba7a          1   

                        super_task_ids_json  \
0  ["cc726cba-2040-4695-a119-159876cc0243"]   
1  ["d55d6f45-c0e1-447a-baaa-ef03c265c780"]   
2  ["e0abd857-c765-4ada-a926-9ffe02e9f2c9"]   

                                                                                                                                                                                                 rows_json  \
0  [{"super_task_id": "cc726cba-2040-4695-a119-159876cc0243", "o_id": "dd2ce4ed-c521-4c6e-a81d-7ff890e9aa06", "v_id": "1", "moment_utc": "2026-03-06 14:28:24.687598", "is_deleted": "False", "body": "...   
1  [{"super_task_id": "d55d6f45-c0e1-447a-baaa-ef03c265c780", "o_id": "2973fcf4-e30b-40b3-b3bd-b437f3b18406", "v_id": "1", "moment_utc": "2026-03-17 01:35:17.553461", "is_deleted": "

## 7. Распарсить `body` из PHR и достать список `hxid` и связанных `Observation`

На этом шаге мы:

- читаем `rows_json` для каждого визита
- внутри каждого `body` ищем `DiagnosticReport`
- достаём из них `hxid`
- находим связанные `Observation`
- сохраняем показатель, коды, результат и статус отклонения

Дополнительно:

- исключаем служебные `Observation` вроде `Врач`, `Референсные значения` и подобных
- не сохраняем `DiagnosticReport`, если после фильтрации у него не осталось ни одного полезного `Observation`
- после построения JSON поднимаем `effectiveDateTime` на уровень визита (и при необходимости сохраняем отдельный файл с датами

Eсли у DiagnosticReport есть specimen, то извлекаем его и добавляем к связанным Observation

In [40]:
EXCLUDED_OBSERVATION_CODE_SYSTEM_PAIRS = {
    ("http://loinc.org", "59256-8"),
    ("http://loinc.org", "19147-8"),
}

EXCLUDED_OBSERVATION_NAMES = {
    "комментарий",
    "врач",
    "название тест-системы",
    "серия тест-системы",
    "срок годности тест-системы",
    "референсные значения",
    "референсные значения по фазам цикла",
}


def parse_iso_datetime(value: str):
    value = (value or "").strip()
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None


def sort_datetimes(values: list[str]) -> list[str]:
    return sorted(
        set(value for value in values if value),
        key=lambda value: (parse_iso_datetime(value) is None, parse_iso_datetime(value), value),
    )


def get_first_display(resource: dict):
    for coding in ((resource.get("code") or {}).get("coding") or []):
        display = (coding.get("display") or "").strip()
        if display:
            return display
    return ((resource.get("code") or {}).get("text") or "").strip()


def get_code_codings(resource: dict):
    codings = []
    for coding in ((resource.get("code") or {}).get("coding") or []):
        code = (coding.get("code") or "").strip()
        system = (coding.get("system") or "").strip()
        display = (coding.get("display") or "").strip()
        if not code and not system and not display:
            continue
        codings.append(
            {
                "code": code,
                "system": system,
                "display": display,
            }
        )
    return codings


def should_skip_observation(obs: dict):
    name = get_first_display(obs).strip().lower()
    if name in EXCLUDED_OBSERVATION_NAMES:
        return True

    for coding in get_code_codings(obs):
        pair = (
            (coding.get("system") or "").strip(),
            (coding.get("code") or "").strip(),
        )
        if pair in EXCLUDED_OBSERVATION_CODE_SYSTEM_PAIRS:
            return True

    return False


def get_hxid(resource: dict):
    for ext in resource.get("extension", []) or []:
        if ext.get("url") == "https://codes/nomenclature":
            return (ext.get("valueString") or "").strip()
    return ""


def get_normal_range(obs: dict):
    for ref_range in obs.get("referenceRange", []) or []:
        meanings = {
            (coding.get("code") or "").strip()
            for coding in ((ref_range.get("type") or {}).get("coding") or [])
            if (coding.get("code") or "").strip()
        }

        if meanings and "normal" not in meanings:
            continue

        low = (ref_range.get("low") or {}).get("value")
        high = (ref_range.get("high") or {}).get("value")
        return low, high

    return None, None


def classify_status(obs: dict, strong_delta: float = 0.5):
    codes = []
    for item in obs.get("interpretation", []) or []:
        for coding in item.get("coding", []) or []:
            code = (coding.get("code") or "").strip().upper()
            if code:
                codes.append(code)

    value_quantity = obs.get("valueQuantity") or {}
    numeric_value = value_quantity.get("value")
    low, high = get_normal_range(obs)

    def high_status():
        if numeric_value is not None and high not in (None, 0):
            delta = (float(numeric_value) - float(high)) / float(high)
            return "HH" if delta >= strong_delta else "H"
        return "H"

    def low_status():
        if numeric_value is not None and low not in (None, 0):
            delta = (float(low) - float(numeric_value)) / float(low)
            return "LL" if delta >= strong_delta else "L"
        return "L"

    code_set = set(codes)

    if {"HH", "HU", ">", "AA"}.intersection(code_set):
        return "HH"
    if {"LL", "LU", "<"}.intersection(code_set):
        return "LL"
    if "H" in code_set:
        return high_status()
    if "L" in code_set:
        return low_status()
    if "N" in code_set:
        return "N"

    if numeric_value is not None:
        if high is not None and float(numeric_value) > float(high):
            return high_status()
        if low is not None and float(numeric_value) < float(low):
            return low_status()
        if low is not None or high is not None:
            return "N"

    return None


def get_observation_value(obs: dict):
    value_quantity = obs.get("valueQuantity") or {}
    if "value" in value_quantity:
        unit = (value_quantity.get("unit") or "").strip()
        value = value_quantity.get("value")
        return f"{value} {unit}".strip()

    value_string = (obs.get("valueString") or "").strip()
    if value_string:
        return value_string

    value_codeable = obs.get("valueCodeableConcept") or {}
    value_text = (value_codeable.get("text") or "").strip()
    if value_text:
        return value_text

    return ""


def build_observation_index(doc: dict):
    by_reference = {}

    for entry in doc.get("entry", []) or []:
        resource = entry.get("resource", {})
        if resource.get("resourceType") != "Observation":
            continue

        full_url = (entry.get("fullUrl") or "").strip()
        resource_id = (resource.get("id") or "").strip()

        if full_url:
            by_reference[full_url] = resource
        if resource_id:
            by_reference[resource_id] = resource
            by_reference[f"Observation/{resource_id}"] = resource

    return by_reference


def build_specimen_index(doc: dict):
    by_reference = {}

    for entry in doc.get("entry", []) or []:
        resource = entry.get("resource", {})
        if resource.get("resourceType") != "Specimen":
            continue

        full_url = (entry.get("fullUrl") or "").strip()
        resource_id = (resource.get("id") or "").strip()

        if full_url:
            by_reference[full_url] = resource
        if resource_id:
            by_reference[resource_id] = resource
            by_reference[f"Specimen/{resource_id}"] = resource

    return by_reference


def get_specimen_display(specimen: dict):
    if not specimen:
        return ""

    for coding in ((specimen.get("type") or {}).get("coding") or []):
        display = (coding.get("display") or "").strip()
        if display:
            return display

    return ((specimen.get("type") or {}).get("text") or "").strip()

def get_container_type(specimen: dict):
    if not specimen:
        return ""

    for container in specimen.get("container", []) or []:
        container_type = container.get("type") or {}
        for coding in container_type.get("coding") or []:
            code = (coding.get("code") or "").strip()
            if code:
                return code

    return ""

def parse_body(body_text: str):
    doc = ast.literal_eval(body_text)

    observation_index = build_observation_index(doc)
    specimen_index = build_specimen_index(doc)

    items = []

    for entry in doc.get("entry", []) or []:
        report = entry.get("resource", {})
        if report.get("resourceType") != "DiagnosticReport":
            continue

        hxid = get_hxid(report)
        if not hxid:
            continue

        report_name = get_first_display(report)
        effective = (report.get("effectiveDateTime") or "").strip()

        specimen_display = ""
        container_type = ""
        for specimen_ref in report.get("specimen", []) or []:
            reference = (specimen_ref.get("reference") or "").strip()
            if not reference:
                continue

            specimen = specimen_index.get(reference)
            if specimen:
                specimen_display = get_specimen_display(specimen)
                container_type = get_container_type(specimen)
                break

        observations = []

        for ref in report.get("result", []) or []:
            reference = (ref.get("reference") or "").strip()
            obs = observation_index.get(reference)

            if not obs:
                continue
            if should_skip_observation(obs):
                continue

            observations.append(
                {
                    "name": get_first_display(obs),
                    "codes": get_code_codings(obs),
                    "result": get_observation_value(obs),
                    "status": classify_status(obs),
                    "specimen": specimen_display,
                    "container_type": container_type,
                    "effectiveDateTime": (
                        obs.get("effectiveDateTime") or ""
                    ).strip(),
                }
            )

        if not observations:
            continue

        items.append(
            {
                "hxid": hxid,
                "report_name": report_name,
                "effectiveDateTime": effective,
                "observations": observations,
            }
        )

    return items


def hoist_effective_datetimes(record: dict):
    visit_dates = []

    for hxid_item in record.get("hxid_items") or []:
        item_dates = []

        for obs in hxid_item.get("observations") or []:
            obs_effective = (obs.get("effectiveDateTime") or "").strip()
            if obs_effective:
                item_dates.append(obs_effective)

            obs.pop("effectiveDateTime", None)

        sorted_item_dates = sort_datetimes(item_dates)

        if sorted_item_dates:
            hxid_item["effectiveDateTime"] = sorted_item_dates[0]
            if len(sorted_item_dates) > 1:
                hxid_item["effectiveDateTime_all"] = sorted_item_dates
        else:
            hxid_item["effectiveDateTime"] = (
                hxid_item.get("effectiveDateTime") or ""
            ).strip()

        if hxid_item.get("effectiveDateTime"):
            visit_dates.append(hxid_item["effectiveDateTime"])

    sorted_visit_dates = sort_datetimes(visit_dates)

    record["effectiveDateTime"] = (
        sorted_visit_dates[0] if sorted_visit_dates else ""
    )

    if len(sorted_visit_dates) > 1:
        record["effectiveDateTime_all"] = sorted_visit_dates

    return record

In [41]:
# Шаг 7. Построение visit_hxid_observations.json (используем aggregated_visits_df)


def _scalar_str(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()

print("Start parsing...")

visit_results = []
for i, row in enumerate(aggregated_visits_df.to_dict("records"), start=1):
    visit_id = _scalar_str(row.get("visit_id"))
    patient_id = _scalar_str(row.get("patient_id"))
    try:
        visit_rows = json.loads(_scalar_str(row.get("rows_json")) or "[]")
    except Exception:
        continue

    hxid_items = []
    for visit_row in visit_rows:
        body_text = visit_row.get("body") or ""
        if not body_text.strip():
            continue
        try:
            hxid_items.extend(parse_body(body_text))
        except Exception:
            continue

    visit_results.append(
        {
            "visit_id": visit_id,
            "patient_id": patient_id,
            "row_count": len(visit_rows),
            "hxid_items": hxid_items,
        }
    )

    if i % 1000 == 0:
        print(f"Scanned visits: {i:,} | parsed visits: {len(visit_results):,}".replace(",", " "))

with open(VISIT_HXID_JSON, "w", encoding="utf-8") as f:
    json.dump(visit_results, f, ensure_ascii=False, indent=2)

print(f"Saved to: {VISIT_HXID_JSON}")

# Подъём effectiveDateTime выше по структуре (конец шага 7).

final_results = []
for record in visit_results:
    record = hoist_effective_datetimes(record)
    final_results.append(record)

with open(VISIT_HXID_WITH_PATIENTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"Saved to: {VISIT_HXID_WITH_PATIENTS_JSON}")

Start parsing...
Scanned visits: 1 000 | parsed visits: 1 000
Scanned visits: 2 000 | parsed visits: 2 000
Scanned visits: 3 000 | parsed visits: 3 000
Scanned visits: 4 000 | parsed visits: 4 000
Scanned visits: 5 000 | parsed visits: 5 000
Scanned visits: 6 000 | parsed visits: 6 000
Scanned visits: 7 000 | parsed visits: 7 000
Scanned visits: 8 000 | parsed visits: 8 000
Scanned visits: 9 000 | parsed visits: 9 000
Scanned visits: 10 000 | parsed visits: 10 000
Scanned visits: 11 000 | parsed visits: 11 000
Scanned visits: 12 000 | parsed visits: 12 000
Scanned visits: 13 000 | parsed visits: 13 000
Scanned visits: 14 000 | parsed visits: 14 000
Scanned visits: 15 000 | parsed visits: 15 000
Scanned visits: 16 000 | parsed visits: 16 000
Scanned visits: 17 000 | parsed visits: 17 000
Scanned visits: 18 000 | parsed visits: 18 000
Scanned visits: 19 000 | parsed visits: 19 000
Saved to: visit_hxid_observations.json
Saved to: visit_hxid_observations_with_patient_ids.json


## Итоговые выходные файлы

- **Шаг 1** пишет на диск `bundle_v2.csv` (сырая выгрузка из PHR).
- **Шаг 2** один раз читает этот CSV; дальше `filtered_bundle_df`, `aggregated_visits_df` и т.д. живут **только в памяти** до конца ноутбука.
- Входные файлы из Metabase (скачиваются вручную): `visits.csv`, `patient_ids.csv`.

На диск в конце пайплайна код сохраняет только JSON:

- `visit_hxid_observations.json` - список `hxid` и связанных `Observation` по визитам (в каждой записи уже есть `patient_id` из шага 6)
- `visit_hxid_observations_with_patient_ids.json` - тот же JSON плюс верхнеуровневый `effectiveDateTime` (и при необходимости `effectiveDateTime_all`)

#### Список типов биоматериала

In [15]:
file_path = "visit_hxid_observations_with_patient_ids.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

specimens = set()

def extract_specimen(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "specimen" and v:
                specimens.add(v)
            else:
                extract_specimen(v)
    elif isinstance(obj, list):
        for item in obj:
            extract_specimen(item)

extract_specimen(data)

print("Unique specimen:", specimens)
print("Total:", len(specimens))

Unique specimen: {'Препарат цитологический', 'Мазок слизистой носоглотки', 'Моча', 'Мазок слизистой носоглотки и ротоглотки', 'Кровь венозная', 'Соскоб с перианальных складок', 'Кровь капиллярная', 'Отделяемое слизистой носа', 'Мазок ректальный', 'Волос', 'Отделяемое слизистой влагалища', 'Мазок слизистой ротоглотки', 'Отделяемое мочеполовых органов', 'Соскоб с эндоцервикса', 'Кал', 'Отделяемое слизистой уретры'}
Total: 16


Маппинг биоматериал - hxid - report_name

In [16]:
file_path = "visit_hxid_observations_with_patient_ids.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

specimen_map = defaultdict(set)

for item in data:
    for hx in item.get("hxid_items", []):
        hxid = hx.get("hxid")
        report_name = hx.get("report_name")

        for obs in hx.get("observations", []):
            specimen = obs.get("specimen")

            if specimen and hxid and report_name:
                specimen_map[specimen].add((hxid, report_name))

for specimen, values in specimen_map.items():
    print(f"\n=== {specimen} ===")
    for hxid, report_name in sorted(values):
        print(f"{hxid} | {report_name}")

print("\nTotal specimens:", len(specimen_map))


=== Моча ===
02-002 | Анализ мочи по Нечипоренко
02-006 | Белок общий в моче
02-006 | Микроскопия мочи
02-006 | Общий анализ мочи
02-032 | Проба Сулковича
06-038 | Белок общий в моче
06-075 | Pyrilinks-D (маркер резорбции костной ткани)
06-075 | Креатинин в моче
06-115 | Глюкоза в моче
06-120 | Селен в моче
06-130 | Ртуть в моче
06-135 | Цинк в моче
06-136 | Медь в моче
06-272 | Йод в моче
09-001 | Candida albicans, ДНК
09-002 | Chlamydia trachomatis, ДНК
09-002 | Mycoplasma genitalium, ДНК
09-002 | Neisseria gonorrhoeae, ДНК
09-002 | Trichomonas vaginalis, ДНК
09-022 | Mycobacterium tuberculosis, ДНК
09-025 | Chlamydia trachomatis, ДНК
09-025 | Mycoplasma genitalium, ДНК
09-025 | Neisseria gonorrhoeae, ДНК
09-025 | Trichomonas vaginalis, ДНК
09-026 | Mycoplasma hominis, ДНК
09-027 | Chlamydia trachomatis, ДНК
09-027 | Mycoplasma genitalium, ДНК
09-027 | Neisseria gonorrhoeae, ДНК
09-027 | Trichomonas vaginalis, ДНК
09-030 | Chlamydia trachomatis, ДНК
09-030 | Mycoplasma genitalium, Д

Выгрузка каталога с правильными report_name по hxid

In [27]:
url = "https://catalog.labpass.online/api/v1/Catalogs/C000153619"
body = {
    "labelsBlackList": [],
    "idBlackList": []
}

all_items = []
last_id = None

while True:
    params = {
        "isDetail": "true",
        "limit": 100
    }

    if last_id:
        params["lastId"] = last_id

    response = requests.post(url, params=params, json=body)
    response.raise_for_status()

    data = response.json()

    items = data.get("items", [])
    all_items.extend(items)

    print(f"Загружено {len(all_items)}")

    last_id = data.get("lastId")

    if len(items) < 100 or not last_id:
        break

with open("catalog.json", "w", encoding="utf-8") as f:
    json.dump(all_items, f, ensure_ascii=False, indent=2)

Загружено 100
Загружено 200
Загружено 300
Загружено 400
Загружено 500
Загружено 600
Загружено 700
Загружено 800
Загружено 900
Загружено 1000
Загружено 1100
Загружено 1200
Загружено 1300
Загружено 1400
Загружено 1500
Загружено 1600
Загружено 1700
Загружено 1800
Загружено 1900
Загружено 2000
Загружено 2095


Формирование итогвого маппинга: биоматериал - hxid - report_name

In [26]:
catalog_path = "catalog.json"
with open(catalog_path, "r", encoding="utf-8") as f:
    catalog = json.load(f)

hxid_to_name = {}
for item in catalog:
    hxid = item.get("labId")
    caption = item.get("caption")
    if hxid and caption:
        hxid_to_name[hxid] = caption

file_path = "visit_hxid_observations_with_patient_ids.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

specimen_map = defaultdict(set)

for item in data:
    for hx in item.get("hxid_items", []):
        hxid = hx.get("hxid")
        report_name = hx.get("report_name")

        # берем название из каталога по hxid
        catalog_name = hxid_to_name.get(hxid)
        name = catalog_name or report_name

        for obs in hx.get("observations", []):
            specimen = obs.get("specimen")

            if specimen and hxid and name:
                specimen_map[specimen].add((hxid, name))

for specimen, values in specimen_map.items():
    print(f"\n=== {specimen} ===")
    for hxid, name in sorted(values):
        print(f"{hxid} | {name}")


=== Моча ===
02-002 | Анализ мочи по Нечипоренко
02-006 | Общий анализ мочи с микроскопией
02-032 | Проба Сулковича
06-038 | Белок общий в моче 
06-075 | Дезоксипиридинолин (Pyrilinks-D) - маркер резорбции костной ткани
06-115 | Глюкоза в моче
06-120 | Селен в моче
06-130 | Ртуть в моче
06-135 | Цинк в моче
06-136 | Медь в моче  
06-272 | Йод в моче
09-001 | Возбудитель кандидоза (Candida albicans), ДНК [реал-тайм ПЦР]
09-002 | Возбудитель хламидиоза (Chlamydia trachomatis), ДНК [реал-тайм ПЦР]
09-022 | Возбудитель туберкулеза (Mycobacterium tuberculosis), ДНК [реал-тайм ПЦР]
09-025 | Возбудитель микоплазмоза (Mycoplasma genitalium), ДНК [реал-тайм ПЦР]
09-026 | Возбудитель микоплазмоза (Mycoplasma hominis), ДНК [реал-тайм ПЦР]
09-027 | Гонококк (Neisseria gonorrhoeae), ДНК [реал-тайм ПЦР]
09-030 | Возбудитель трихомоноза (Trichomonas vaginalis), ДНК [реал-тайм ПЦР]
09-032 | Уреаплазма уреалитикум (Ureaplasma urealyticum), ДНК [реал-тайм ПЦР]
09-121 | Микоплазмы (Mycoplasma spp.), ДНК

Формирование маппинга: биоматериал - тип пробирки - hxid - report_name

In [44]:
catalog_path = "catalog.json"
with open(catalog_path, "r", encoding="utf-8") as f:
    catalog = json.load(f)

hxid_to_name = {}
for item in catalog:
    hxid = item.get("labId")
    caption = item.get("caption")
    if hxid and caption:
        hxid_to_name[hxid] = caption

file_path = "visit_hxid_observations_with_patient_ids.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
raw_map = defaultdict(lambda: defaultdict(set))

for item in data:
    for hx in item.get("hxid_items", []):
        hxid = hx.get("hxid")
        report_name = hx.get("report_name")

        catalog_name = hxid_to_name.get(hxid)
        name = catalog_name or report_name

        if not hxid or not name:
            continue

        for obs in hx.get("observations", []):
            specimen = (obs.get("specimen") or "").strip()
            container_type = (obs.get("container_type") or "").strip()

            if specimen and container_type:
                raw_map[specimen][container_type].add((hxid, name.strip()))

output = {}
for specimen, containers in raw_map.items():
    output[specimen] = {}
    for container_type, items in containers.items():
        output[specimen][container_type] = [
            {"hxid": h, "report_name": n}
            for h, n in sorted(items)
        ]

out_path = "specimen_container_catalog_mapping.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

In [ ]:
{
  "Моча": [
    {"hxid": "02-002", "report_name": "Анализ мочи по Нечипоренко"},
    {"hxid": "02-006", "report_name": "Общий анализ мочи с микроскопией"},
    {"hxid": "02-032", "report_name": "Проба Сулковича"},
    {"hxid": "06-038", "report_name": "Белок общий в моче"},
    {"hxid": "06-075", "report_name": "Дезоксипиридинолин (Pyrilinks-D) - маркер резорбции костной ткани"},
    {"hxid": "06-115", "report_name": "Глюкоза в моче"},
    {"hxid": "06-120", "report_name": "Селен в моче"},
    {"hxid": "06-130", "report_name": "Ртуть в моче"},
    {"hxid": "06-135", "report_name": "Цинк в моче"},
    {"hxid": "06-136", "report_name": "Медь в моче"},
    {"hxid": "06-272", "report_name": "Йод в моче"},
    {"hxid": "09-001", "report_name": "Возбудитель кандидоза (Candida albicans), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-002", "report_name": "Возбудитель хламидиоза (Chlamydia trachomatis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-022", "report_name": "Возбудитель туберкулеза (Mycobacterium tuberculosis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-025", "report_name": "Возбудитель микоплазмоза (Mycoplasma genitalium), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-026", "report_name": "Возбудитель микоплазмоза (Mycoplasma hominis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-027", "report_name": "Гонококк (Neisseria gonorrhoeae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-030", "report_name": "Возбудитель трихомоноза (Trichomonas vaginalis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-032", "report_name": "Уреаплазма уреалитикум (Ureaplasma urealyticum), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-121", "report_name": "Микоплазмы (Mycoplasma spp.), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-172", "report_name": "Развернутая диагностика ЗППП для мужчин (\"Андрофлор\"), ДНК количественно [реал-тайм ПЦР]"},
    {"hxid": "09-173", "report_name": "\"Андрофлор-скрин\", ДНК количественно [реал-тайм ПЦР]"},
    {"hxid": "09-175", "report_name": "Уреаплазма парвум (Ureaplasma parvum), ДНК [реал-тайм ПЦР],количественно"},
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "10-043", "report_name": "Посев на анаэробную флору"},
    {"hxid": "10-049", "report_name": "Посев на аэробную и факультативно-анаэробную флору"},
    {"hxid": "12-011", "report_name": "Цитологическое исследование мочи"},
    {"hxid": "40-110", "report_name": "Интимный - 8 тестов по моче"},
    {"hxid": "40-505", "report_name": "Альбумин-креатининовое соотношение (альбуминурия в разовой порции мочи)"},
    {"hxid": "40-647", "report_name": "Комплексная оценка риска камнеобразования - литогенные субстанции мочи"}
  ],
  "Кровь венозная": [
    {"hxid": "02-005", "report_name": "Клинический анализ крови (с лейкоцитарной формулой)"},
    {"hxid": "02-007", "report_name": "Скорость оседания эритроцитов (СОЭ)"},
    {"hxid": "02-014", "report_name": "Общий анализ крови (без лейкоцитарной формулы и СОЭ)"},
    {"hxid": "02-025", "report_name": "Лейкоцитарная формула (с микроскопией мазка крови при выявлении патологических изменений)"},
    {"hxid": "02-027", "report_name": "Ретикулоциты"},
    {"hxid": "02-029", "report_name": "Клинический анализ крови: общий анализ, лейкоцитарная формула, СОЭ (с микроскопией мазка крови при выявлении патологических изменений)"},
    {"hxid": "02-041", "report_name": "Клинический анализ крови с микроскопией лейкоцитарной формулы"},
    {"hxid": "02-042", "report_name": "Лейкоцитарная формула (с обязательной микроскопией мазка крови)"},
    {"hxid": "02-043", "report_name": "Клинический анализ крови: общий анализ, лейкоцитарная формула, СОЭ (с обязательной микроскопией мазка крови)"},
    {"hxid": "02-058", "report_name": "Карбоксигемоглобин и метгемоглобин в крови"},
    {"hxid": "02-059", "report_name": "Карбоксигемоглобин в крови"},
    {"hxid": "03-001", "report_name": "D-димер"},
    {"hxid": "03-002", "report_name": "Антитромбин III"},
    {"hxid": "03-003", "report_name": "Активированное частичное тромбопластиновое время (АЧТВ)"},
    {"hxid": "03-004", "report_name": "Волчаночный антикоагулянт"},
    {"hxid": "03-005", "report_name": "Группа крови ABO"},
    {"hxid": "03-007", "report_name": "Коагулограмма №1 (протромбин (по Квику), МНО)"},
    {"hxid": "03-008", "report_name": "Резус-фактор"},
    {"hxid": "03-010", "report_name": "Тромбиновое время"},
    {"hxid": "03-011", "report_name": "Фибриноген"},
    {"hxid": "03-013", "report_name": "Эритропоэтин"},
    {"hxid": "03-015", "report_name": "Коагулограмма №2 (протромбин (по Квику), МНО, фибриноген)"},
    {"hxid": "03-016", "report_name": "Коагулограмма №3 (протромбин (по Квику), МНО, фибриноген, АТIII, АЧТВ, D-димер)"},
    {"hxid": "03-018", "report_name": "Протеин C"},
    {"hxid": "03-019", "report_name": "Протеин S свободный"},
    {"hxid": "06-001", "report_name": "С-концевые телопептиды коллагена I типа (β-CrossLaps) - маркер резорбции костной ткани"},
    {"hxid": "06-002", "report_name": "N-остеокальцин - маркер костного ремоделирования"},
    {"hxid": "06-003", "report_name": "Аланинаминотрансфераза (АЛТ)"},
    {"hxid": "06-004", "report_name": "Альбумин в сыворотке"},
    {"hxid": "06-005", "report_name": "Амилаза общая в сыворотке"},
    {"hxid": "06-006", "report_name": "Амилаза панкреатическая"},
    {"hxid": "06-007", "report_name": "Антистрептолизин О"},
    {"hxid": "06-008", "report_name": "Аполипопротеин B"},
    {"hxid": "06-009", "report_name": "Аполипопротеин A1"},
    {"hxid": "06-010", "report_name": "Аспартатаминотрансфераза (АСТ)"},
    {"hxid": "06-011", "report_name": "Белковые фракции в сыворотке"},
    {"hxid": "06-012", "report_name": "Витамин B12 (цианокобаламин)"},
    {"hxid": "06-013", "report_name": "Гамма-глютамилтранспептидаза (гамма-ГТ)"},
    {"hxid": "06-014", "report_name": "Гликированный гемоглобин (HbA1c)"},
    {"hxid": "06-015", "report_name": "Глюкоза в плазме"},
    {"hxid": "06-016", "report_name": "Гомоцистеин"},
    {"hxid": "06-017", "report_name": "Железо в сыворотке"},
    {"hxid": "06-018", "report_name": "Железосвязывающая способность сыворотки"},
    {"hxid": "06-019", "report_name": "Калий, натрий, хлор в сыворотке"},
    {"hxid": "06-020", "report_name": "Кальций в сыворотке"},
    {"hxid": "06-021", "report_name": "Креатинин в сыворотке (с определением СКФ)"},
    {"hxid": "06-022", "report_name": "Креатинкиназа общая"},
    {"hxid": "06-023", "report_name": "Креатинкиназа MB"},
    {"hxid": "06-024", "report_name": "Лактат"},
    {"hxid": "06-025", "report_name": "Лактатдегидрогеназа (ЛДГ) общая"},
    {"hxid": "06-026", "report_name": "Лактатдегидрогеназа 1, 2 (ЛДГ 1, 2 фракции)"},
    {"hxid": "06-027", "report_name": "Липаза"},
    {"hxid": "06-028", "report_name": "Холестерол – липопротеины высокой плотности (ЛПВП)"},
    {"hxid": "06-029", "report_name": "Холестерол – липопротеины низкой плотности (ЛПНП)"},
    {"hxid": "06-031", "report_name": "Магний в сыворотке"},
    {"hxid": "06-033", "report_name": "Мочевая кислота в сыворотке"},
    {"hxid": "06-034", "report_name": "Мочевина в сыворотке"},
    {"hxid": "06-035", "report_name": "Белок общий в сыворотке"},
    {"hxid": "06-036", "report_name": "Билирубин общий"},
    {"hxid": "06-037", "report_name": "Билирубин прямой"},
    {"hxid": "06-039", "report_name": "С-пептид в сыворотке"},
    {"hxid": "06-040", "report_name": "Трансферрин"},
    {"hxid": "06-041", "report_name": "Триглицериды"},
    {"hxid": "06-042", "report_name": "Ферритин"},
    {"hxid": "06-043", "report_name": "Витамин B9 (фолиевая кислота)"},
    {"hxid": "06-045", "report_name": "Фосфатаза щелочная общая"},
    {"hxid": "06-046", "report_name": "Фосфор в сыворотке"},
    {"hxid": "06-047", "report_name": "Фруктозамин"},
    {"hxid": "06-048", "report_name": "Холестерол общий"},
    {"hxid": "06-049", "report_name": "Холинэстераза в сыворотке"},
    {"hxid": "06-050", "report_name": "С-реактивный белок, количественно (высокочувствительный метод)"},
    {"hxid": "06-051", "report_name": "Кальций ионизированный"},
    {"hxid": "06-064", "report_name": "Калий в сыворотке"},
    {"hxid": "06-065", "report_name": "Натрий в сыворотке"},
    {"hxid": "06-066", "report_name": "Хлор в сыворотке"},
    {"hxid": "06-076", "report_name": "Тропонин I"},
    {"hxid": "06-077", "report_name": "Гаптоглобин"},
    {"hxid": "06-078", "report_name": "Альфа-1-антитрипсин"},
    {"hxid": "06-079", "report_name": "Миоглобин"},
    {"hxid": "06-080", "report_name": "Церулоплазмин"},
    {"hxid": "06-082", "report_name": "Цинк в сыворотке"},
    {"hxid": "06-083", "report_name": "Медь в плазме"},
    {"hxid": "06-084", "report_name": "Литий в плазме"},
    {"hxid": "06-087", "report_name": "Кремний в плазме"},
    {"hxid": "06-089", "report_name": "Хром в плазме"},
    {"hxid": "06-090", "report_name": "Марганец в плазме"},
    {"hxid": "06-091", "report_name": "Кобальт в плазме"},
    {"hxid": "06-094", "report_name": "Селен в плазме"},
    {"hxid": "06-095", "report_name": "Молибден в плазме"},
    {"hxid": "06-098", "report_name": "Ртуть в плазме"},
    {"hxid": "06-101", "report_name": "Витамин А (ретинол)"},
    {"hxid": "06-102", "report_name": "Витамин B1 (тиамин)"},
    {"hxid": "06-103", "report_name": "Витамин B5 (пантотеновая кислота)"},
    {"hxid": "06-104", "report_name": "Витамин B6 (пиридоксаль-5-фосфат)"},
    {"hxid": "06-105", "report_name": "Витамин С (аскорбиновая кислота)"},
    {"hxid": "06-106", "report_name": "Витамин D, 25-гидрокси (кальциферол)"},
    {"hxid": "06-107", "report_name": "Витамин Е (токоферол)"},
    {"hxid": "06-108", "report_name": "Витамин К (филлохинон)"},
    {"hxid": "06-112", "report_name": "Комплексный анализ крови на ненасыщенные жирные кислоты семейства омега-3"},
    {"hxid": "06-133", "report_name": "Латентная железосвязывающая способность сыворотки"},
    {"hxid": "06-157", "report_name": "Мозговой натрийуретический гормон (NT-proBNP), количественно"},
    {"hxid": "06-178", "report_name": "Липопротеин (a)"},
    {"hxid": "06-180", "report_name": "Фосфатаза кислая общая"},
    {"hxid": "06-182", "report_name": "С-реактивный белок, количественно (метод с нормальной чувствительностью)"},
    {"hxid": "06-192", "report_name": "Анализ крови на органические кислоты"},
    {"hxid": "06-217", "report_name": "Витамин B2 (рибофлавин)"},
    {"hxid": "06-218", "report_name": "Витамин B3 (ниацин, никотинамид)"},
    {"hxid": "06-219", "report_name": "Комплексный анализ крови на витамины группы D (D2 и D3)"},
    {"hxid": "06-220", "report_name": "Определение омега-3-индекса"},
    {"hxid": "06-228", "report_name": "Расширенный комплексный анализ на витамины (A, бета-каротин, D, E, K, C, B1, B2, B3, B5, B6, B9, B12)"},
    {"hxid": "06-229", "report_name": "Комплексный анализ на витамины группы B (B1, B2, B3, B5, B6, B7, B9, B12)"},
    {"hxid": "06-233", "report_name": "Основные эссенциальные (жизненно необходимые) и токсичные микроэлементы (13 показателей)"},
    {"hxid": "06-234", "report_name": "Комплексный анализ на наличие тяжелых металлов и микроэлементов (23 показателя)"},
    {"hxid": "06-246", "report_name": "Витамины и микроэлементы, влияющие на состояние мышечной системы (K, Na, Ca, Mg, Zn, Mn, витамины B1, B5)"},
    {"hxid": "06-251", "report_name": "Витамины и микроэлементы, участвующие в регуляции функции щитовидной железы (I, Se, Mg, Cu, витамин B6)"},
    {"hxid": "06-253", "report_name": "Витамины и микроэлементы, участвующие в регуляции выделительной системы (K, Na, Ca, Mg, витамины B6, D)"},
    {"hxid": "06-256", "report_name": "Альфа-2-макроглобулин"},
    {"hxid": "06-261", "report_name": "Прокальцитонин"},
    {"hxid": "06-263", "report_name": "Цистатин C"},
    {"hxid": "06-264", "report_name": "Расширенный комплексный анализ крови на метаболиты витамина D (1,25-OH D3, 25-OH D3, 25-OH D2, 24,25-OH D3)"},
    {"hxid": "06-265", "report_name": "Йод в плазме"},
    {"hxid": "06-268", "report_name": "Анализ крови на аминокислоты (13 показателей)"},
    {"hxid": "06-270", "report_name": "Растворимые рецепторы трансферрина"},
    {"hxid": "06-275", "report_name": "Глутатионпероксидаза в эритроцитах"},
    {"hxid": "06-277", "report_name": "Витамин B7 (биотин)"},
    {"hxid": "06-311", "report_name": "Витамин B12, активный (холотранскобаламин)"},
    {"hxid": "06-312", "report_name": "Витамин D, 25-гидрокси (кальциферол), ВЭЖХ-МС/МС"},
    {"hxid": "06-313", "report_name": "Витамин B2 (ФАД)"},
    {"hxid": "06-334", "report_name": "Сера в плазме"},
    {"hxid": "06-355", "report_name": "Анализ крови на ацилкарнитины (взрослые)"},
    {"hxid": "06-425", "report_name": "Рений в плазме"},
    {"hxid": "06-441", "report_name": "Витамин К2"},
    {"hxid": "06-444", "report_name": "Окисленные липопротеины низкой плотности (ОкЛНП)"},
    {"hxid": "06-449", "report_name": "Тропонин I высокочувствительный"},
    {"hxid": "07-002", "report_name": "Антитела к вирусу гепатита A  (anti-HAV, IgM)"},
    {"hxid": "07-004", "report_name": "Антитела к ядерному антигену вируса гепатита B (anti-HBc, IgM)"},
    {"hxid": "07-005", "report_name": "Антитела к ядерному антигену вируса гепатита B (anti-HBc), суммарные"},
    {"hxid": "07-007", "report_name": "Антитела к поверхностному антигену вируса гепатита B (anti-HBs), суммарные"},
    {"hxid": "07-009", "report_name": "Антитела к вирусу гепатита C (anti-HCV), суммарные"},
    {"hxid": "07-010", "report_name": "Антитела к структурным и неструктурным белкам вируса гепатита С"},
    {"hxid": "07-011", "report_name": "Антитела к плесневым грибам (Aspergillus fumigatus, IgG)"},
    {"hxid": "07-012", "report_name": "Антитела к дрожжеподобным грибам (Candida albicans, IgG)"},
    {"hxid": "07-013", "report_name": "Антитела к хламидии трахоматис (Chlamydia trachomatis, IgA)"},
    {"hxid": "07-014", "report_name": "Антитела к хламидии трахоматис (Chlamydia trachomatis, IgG)"},
    {"hxid": "07-015", "report_name": "Антитела к хламидии трахоматис (Chlamydia trachomatis, IgM)"},
    {"hxid": "07-016", "report_name": "Антитела к предраннему белку цитомегаловируса (CMV IEA, IgM/IgG)"},
    {"hxid": "07-017", "report_name": "Антитела к цитомегаловирусу (Cytomegalovirus, IgG)"},
    {"hxid": "07-018", "report_name": "Антитела к цитомегаловирусу (Cytomegalovirus, IgM)"},
    {"hxid": "07-019", "report_name": "Антитела к эхинококку (Echinococcus, IgG)"},
    {"hxid": "07-020", "report_name": "Антитела к капсидному белку вируса Эпштейна-Барр (EBV VCA, IgM)"},
    {"hxid": "07-021", "report_name": "Антитела к ранним антигенам вируса Эпштейна-Барр (EBV EA, IgG)"},
    {"hxid": "07-022", "report_name": "Антитела к ядерному антигену вируса Эпштейна-Барр (EBNA, IgG), количественно"},
    {"hxid": "07-023", "report_name": "Антитела к лямблиям (Giardia lamblia), суммарные"},
    {"hxid": "07-024", "report_name": "Е-антиген вируса гепатита B (HBeAg)"},
    {"hxid": "07-025", "report_name": "Поверхностный антиген вируса гепатита B (HBsAg)"},
    {"hxid": "07-027", "report_name": "Антитела к хеликобактеру пилори (Helicobacter pylori, IgA)"},
    {"hxid": "07-028", "report_name": "Антитела к хеликобактеру пилори (Helicobacter pylori, IgG), количественно"},
    {"hxid": "07-030", "report_name": "Антитела к вирусу простого герпеса (HSV 1/2, IgG)"},
    {"hxid": "07-031", "report_name": "Антитела к вирусу простого герпеса (HSV 1/2, IgM)"},
    {"hxid": "07-032", "report_name": "Антитела к ВИЧ 1/2 и антигену p24 (HIV Ag/Ab)"},
    {"hxid": "07-033", "report_name": "Антитела к вирусу кори (Measles Virus, IgG)"},
    {"hxid": "07-036", "report_name": "Антитела к микоплазме хоминис (Mycoplasma hominis, IgG), титр"},
    {"hxid": "07-037", "report_name": "Антитела к описторхам (Opisthorchis, IgG)"},
    {"hxid": "07-040", "report_name": "Антитела к вирусу краснухи (Rubella, IgG), количественно"},
    {"hxid": "07-042", "report_name": "Антитела к вирусу краснухи (Rubella, IgM)"},
    {"hxid": "07-043", "report_name": "Антитела к токсокаре (Toxocara, IgG), титр"},
    {"hxid": "07-044", "report_name": "Антитела к токсоплазме гонди (Toxoplasma gondii, IgG), количественно"},
    {"hxid": "07-046", "report_name": "Антитела к токсоплазме гонди (Toxoplasma gondii, IgM)"},
    {"hxid": "07-047", "report_name": "Антитела к бледной трепонеме (Treponema pallidum, IgG), титр"},
    {"hxid": "07-048", "report_name": "Антитела к бледной трепонеме (Treponema pallidum, IgM), титр"},
    {"hxid": "07-049", "report_name": "Антитела к бледной трепонеме (Treponema pallidum)"},
    {"hxid": "07-050", "report_name": "Антитела к трихинелле (Trichinella, IgG)"},
    {"hxid": "07-052", "report_name": "Антитела к уреаплазме уреалитикум (Ureaplasma urealyticum, IgA)"},
    {"hxid": "07-054", "report_name": "Антитела к вирусу Варицелла-Зостер (VZV, IgG)"},
    {"hxid": "07-055", "report_name": "Антитела к вирусу Варицелла-Зостер (VZV, IgM)"},
    {"hxid": "07-056", "report_name": "Сифилис RPR (антикардиолипиновый тест/микрореакция преципитации), титр"},
    {"hxid": "07-064", "report_name": "Антитела к боррелии бургдорфери (Borrelia burgdorferi, IgG)"},
    {"hxid": "07-076", "report_name": "Антитела к хламидии пневмониа (Chlamydia pneumoniae, IgM)"},
    {"hxid": "07-077", "report_name": "Антитела к хламидии пневмониа (Chlamydia pneumoniae, IgG)"},
    {"hxid": "07-078", "report_name": "Антитела к микоплазме пневмониа (Mycoplasma pneumoniae, IgM)"},
    {"hxid": "07-079", "report_name": "Антитела к микоплазме пневмониа (Mycoplasma pneumoniae, IgA)"},
    {"hxid": "07-080", "report_name": "Антитела к микоплазме пневмониа (Mycoplasma pneumoniae, IgG)"},
    {"hxid": "07-082", "report_name": "Антитела к вирусу герпеса (HHV 6, IgG)"},
    {"hxid": "07-086", "report_name": "Антитела к возбудителю коклюша и паракоклюша (Bordetella pertussis/parapertussis)"},
    {"hxid": "07-093", "report_name": "Антитела к вирусу клещевого энцефалита (TBEV, IgG)"},
    {"hxid": "07-097", "report_name": "Антитела к вирусу эпидемического паротита (Mumps Virus, IgG)"},
    {"hxid": "07-098", "report_name": "Антитела к хламидии пневмониа (Chlamydia pneumoniae, IgA)"},
    {"hxid": "07-099", "report_name": "Антитела к сероварам A, B, C1, C2, D, E сальмонелл (Salmonella spp.)"},
    {"hxid": "07-101", "report_name": "Антитела к вирусу гепатита C (anti-HCV), ИФА"},
    {"hxid": "07-102", "report_name": "Антитела к бледной трепонеме (Treponema pallidum), ИФА"},
    {"hxid": "07-104", "report_name": "Антитела к возбудителю столбняка (Clostridium tetani, IgG)"},
    {"hxid": "07-107", "report_name": "Антитела к возбудителям шигеллеза (Shigella flexneri 1-5/6/sonnei)"},
    {"hxid": "07-108", "report_name": "Антитела к Vi-aнтигену сальмонеллы (Salmonella typhi)"},
    {"hxid": "07-110", "report_name": "Антитела к возбудителю дифтерии (Corynebacterium diphtheriae)"},
    {"hxid": "07-111", "report_name": "Антитела к HBе-антигену вируса гепатита B (anti-Hbe)"},
    {"hxid": "07-113", "report_name": "Скрининговое обследование на гельминтозы (Opisthorchis, Toxocara, Trichinella, Echinococcus)"},
    {"hxid": "07-115", "report_name": "Антитела к возбудителям псевдотуберкулеза и иерсиниоза (Yersinia pseudotuberculosis/enterocolitica, IgM), качественно"},
    {"hxid": "07-117", "report_name": "Сифилис РПГА (реакция пассивной гемагглютинации), титр"},
    {"hxid": "07-123", "report_name": "Антитела к возбудителю аскаридоза (Ascaris lumbricoides, IgG)"},
    {"hxid": "07-124", "report_name": "Антитела к капсидному белку вируса Эпштейна-Барр (EBV VCA, IgG)"},
    {"hxid": "07-125", "report_name": "Антитела к лямблии (Giardia lamblia, IgM)"},
    {"hxid": "07-131", "report_name": "Антитела к Т-лимфотропному вирусу человека типа 1 и 2 (HTLV-1/2)"},
    {"hxid": "07-132", "report_name": "Антитела к цистицеркам свиного цепня (Taenia solium, IgG )"},
    {"hxid": "07-133", "report_name": "Антитела к анизакидам (Anisakis, IgG)"},
    {"hxid": "07-134", "report_name": "Антитела к вирусу простого герпеса (HSV 1, IgG)"},
    {"hxid": "07-135", "report_name": "Антитела к вирусу простого герпеса (HSV 2, IgG)"},
    {"hxid": "07-138", "report_name": "Антитела к токсоплазме гонди (Toxoplasma gondii, IgА)"},
    {"hxid": "07-140", "report_name": "Антитела к китайской двуустке (Clonorchis sinensis, IgG)"},
    {"hxid": "07-141", "report_name": "Антитела к хеликобактеру пилори (Helicobacter pylori, IgM)"},
    {"hxid": "07-143", "report_name": "Антитела к вирусу гепатита А (anti-HAV)"},
    {"hxid": "07-144", "report_name": "Антитела к иерсинии псевдотуберкулезной (Yersinia pseudotuberculosis), РНГА"},
    {"hxid": "07-145", "report_name": "Антитела к серотипам O:3 и O:9 иерсинии кишечной (Yersinia enterocolitica), РНГА"},
    {"hxid": "07-148", "report_name": "Антитела к вирусу гепатита D (anti-HDV, IgM)"},
    {"hxid": "07-151", "report_name": "Антитела к коклюшной палочке (Bordetella pertussis, IgM)"},
    {"hxid": "07-152", "report_name": "Антитела к коклюшной палочке (Bordetella pertussis, IgG)"},
    {"hxid": "07-158", "report_name": "Антитела к амебе дизентерийной (Entamoeba histolytica, IgG)"},
    {"hxid": "07-159", "report_name": "Антитела к угрице кишечной (Strongyloides stercoralis, IgG)"},
    {"hxid": "07-160", "report_name": "Определение авидности иммуноглобулинов к цитомегаловирусу (Cytomegalovirus, IgG)"},
    {"hxid": "07-161", "report_name": "Определение авидности иммуноглобулинов к вирусу Эпштейн-Барр (EBV VCA, IgG)"},
    {"hxid": "07-163", "report_name": "Определение авидности иммуноглобулинов к вирусу краснухи (Rubella virus, IgG)"},
    {"hxid": "07-165", "report_name": "Иммунологическая диагностика туберкулёза методом T-SPOT.TB"},
    {"hxid": "07-170", "report_name": "Антитела к белку теплового шока хламидии (Chlamydia trachomatis) (anti-cHSP60, IgG)"},
    {"hxid": "07-174", "report_name": "Антитела к вирусу гепатита C (anti-HCV, IgM)"},
    {"hxid": "07-186", "report_name": "Антитела к описторхам (Opisthorchis, IgM)"},
    {"hxid": "07-187", "report_name": "ЦИК, содержащие антигены описторхов"},
    {"hxid": "07-194", "report_name": "Антитела к дрожжеподобным грибам (Candida albicans, IgA)"},
    {"hxid": "07-195", "report_name": "Антитела к дрожжеподобным грибам (Candida albicans, IgM)"},
    {"hxid": "07-196", "report_name": "Антитела к парвовирусу (Parvovirus B19, IgG)"},
    {"hxid": "07-197", "report_name": "Антитела к парвовирусу (Parvovirus B19, IgM)"},
    {"hxid": "07-198", "report_name": "Антитела к бруцеллам (Brucella, IgА)"},
    {"hxid": "07-199", "report_name": "Антитела к бруцеллам (Brucella, IgG)"},
    {"hxid": "07-200", "report_name": "Поверхностный антиген вируса гепатита В (HBsAg), количественно"},
    {"hxid": "07-203", "report_name": "Антитела к главному белку наружной мембраны МОМР и мембраноассоциированному плазмидному белку Pgp3 Chlamydia trachomatis, IgG"},
    {"hxid": "07-204", "report_name": "Антитела к вирусу герпеса (HHV 8, IgG)"},
    {"hxid": "07-208", "report_name": "Антитела к коронавирусу SARS-CoV-2 (COVID19), к спайковому (S) и нуклеокапсидному (N) белкам, IgG, качественно"},
    {"hxid": "07-209", "report_name": "Антитела к коронавирусу SARS-CoV-2 (COVID-19), IgM, качественно"},
    {"hxid": "07-223", "report_name": "Антитела к возбудителям бруцеллеза (Brucella spp.)"},
    {"hxid": "07-226", "report_name": "Антитела к коклюшной палочке (Bordetella pertussis, IgA), количественно"},
    {"hxid": "07-227", "report_name": "Диагностика туберкулеза (ТВ-Feron IGRA)"},
    {"hxid": "08-003", "report_name": "17-гидроксипрогестерон (17-ОПГ)"},
    {"hxid": "08-004", "report_name": "Раковый антиген CA 125"},
    {"hxid": "08-005", "report_name": "Раковый антиген CA 15-3"},
    {"hxid": "08-006", "report_name": "Раковый антиген CA 19-9"},
    {"hxid": "08-007", "report_name": "Раковый антиген CA 72-4"},
    {"hxid": "08-008", "report_name": "Фрагменты цитокератина 19 CYFRA 21-1"},
    {"hxid": "08-009", "report_name": "Суммарные иммуноглобулины A (IgA) в сыворотке"},
    {"hxid": "08-010", "report_name": "Суммарные иммуноглобулины G (IgG) в сыворотке"},
    {"hxid": "08-011", "report_name": "Суммарные иммуноглобулины M (IgM) в сыворотке"},
    {"hxid": "08-012", "report_name": "Адренокортикотропный гормон (АКТГ)"},
    {"hxid": "08-013", "report_name": "Альдостерон"},
    {"hxid": "08-014", "report_name": "Андростендион"},
    {"hxid": "08-016", "report_name": "Альфа-фетопротеин (альфа-ФП)"},
    {"hxid": "08-017", "report_name": "Суммарные иммуноглобулины E (IgE) в сыворотке"},
    {"hxid": "08-018", "report_name": "Бета-2-микроглобулин в сыворотке"},
    {"hxid": "08-020", "report_name": "Бета-субъединица хорионического гонадотропина человека (бета-ХГЧ)"},
    {"hxid": "08-021", "report_name": "Свободная бета-субъединица хорионического гонадотропина человека (бета-ХГЧ свободный)"},
    {"hxid": "08-023", "report_name": "Глобулин, связывающий половые гормоны (ГСПГ)"},
    {"hxid": "08-024", "report_name": "Дигидротестостерон"},
    {"hxid": "08-026", "report_name": "Инсулин"},
    {"hxid": "08-027", "report_name": "Кальцитонин в сыворотке"},
    {"hxid": "08-030", "report_name": "Кортизол"},
    {"hxid": "08-033", "report_name": "Паратиреоидный гормон, интактный"},
    {"hxid": "08-038", "report_name": "Простат-специфический антиген общий (ПСА общий)"},
    {"hxid": "08-042", "report_name": "Раковый эмбриональный антиген (РЭА)"},
    {"hxid": "08-043", "report_name": "Соматотропный гормон"},
    {"hxid": "08-050", "report_name": "Тестостерон свободный"},
    {"hxid": "08-051", "report_name": "Тиреоглобулин"},
    {"hxid": "08-056", "report_name": "Эстриол свободный"},
    {"hxid": "08-058", "report_name": "Нейронспецифическая энолаза (NSE)"},
    {"hxid": "08-071", "report_name": "Гастрин"},
    {"hxid": "08-083", "report_name": "Катехоламины (адреналин, норадреналин, дофамин) и серотонин в крови"},
    {"hxid": "08-085", "report_name": "Инсулиноподобный фактор роста"},
    {"hxid": "08-089", "report_name": "Ингибин B"},
    {"hxid": "08-091", "report_name": "Муциноподобный рако-ассоциированный антиген (MCA)"},
    {"hxid": "08-093", "report_name": "Антимюллеровский гормон"},
    {"hxid": "08-094", "report_name": "Эозинофильный катионный белок (ECP), ImmunoCAP"},
    {"hxid": "08-095", "report_name": "Ренин"},
    {"hxid": "08-096", "report_name": "Лептин"},
    {"hxid": "08-097", "report_name": "Пепсиноген I"},
    {"hxid": "08-105", "report_name": "Раковый антиген CA 242"},
    {"hxid": "08-110", "report_name": "Дегидроэпиандростеронсульфат (ДЭА-SO4)"},
    {"hxid": "08-111", "report_name": "Лютеинизирующий гормон (ЛГ)"},
    {"hxid": "08-112", "report_name": "Прогестерон"},
    {"hxid": "08-113", "report_name": "Трийодтиронин общий (Т3)"},
    {"hxid": "08-114", "report_name": "Трийодтиронин свободный (Т3 свободный)"},
    {"hxid": "08-115", "report_name": "Тироксин общий (Т4)"},
    {"hxid": "08-116", "report_name": "Тироксин свободный (Т4 свободный)"},
    {"hxid": "08-117", "report_name": "Тестостерон"},
    {"hxid": "08-118", "report_name": "Тиреотропный гормон (ТТГ)"},
    {"hxid": "08-119", "report_name": "Фолликулостимулирующий гормон (ФСГ)"},
    {"hxid": "08-120", "report_name": "Эстрадиол"},
    {"hxid": "08-121", "report_name": "Макропролактин"},
    {"hxid": "08-122", "report_name": "Пролактин"},
    {"hxid": "08-124", "report_name": "Андростендиол глюкуронид"},
    {"hxid": "08-125", "report_name": "Проинсулин"},
    {"hxid": "08-129", "report_name": "Гастрин-17"},
    {"hxid": "08-134", "report_name": "Триптаза"},
    {"hxid": "08-135", "report_name": "Белок S-100"},
    {"hxid": "08-137", "report_name": "Человеческий эпидидимальный протеин 4 (HE4)"},
    {"hxid": "08-139", "report_name": "Плацентарный фактор роста (PlGF)"},
    {"hxid": "08-141", "report_name": "Хромогранин А"},
    {"hxid": "08-151", "report_name": "Альдостерон-рениновое соотношение"},
    {"hxid": "08-164", "report_name": "Эстрогены в крови (3 показателя)"},
    {"hxid": "08-168", "report_name": "Тироксин свободный (Т4 свободный), ВЭЖХ"},
    {"hxid": "08-172", "report_name": "Серотонин в крови"},
    {"hxid": "08-175", "report_name": "Андрогены, глюкокортикоиды, минералокортикоиды, эстрогены, прогестагены, их предшественники и метаболиты (18 показателей) в сыворотке"},
    {"hxid": "08-179", "report_name": "Биогенные амины: адреналин, норадреналин, дофамин, серотонин и их метилированные метаболиты (метанефрин, норметанефрин) в сыворотке"},
    {"hxid": "08-180", "report_name": "Метилированные производные аргинина: монометиларгинин (MMA), асимметричный диметиларгинин (ADMA), симметричный диметиларгинин (SDMA)"},
    {"hxid": "08-182", "report_name": "Эозинофильный катионный белок"},
    {"hxid": "08-184", "report_name": "Тестостерон общий в крови, ВЭЖХ"},
    {"hxid": "09-003", "report_name": "Цитомегаловирус (Cytomegalovirus), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-006", "report_name": "Вирус Эпштейн-Барр (Epstein Barr Virus), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-008", "report_name": "Вирус гепатита В (HBV), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-009", "report_name": "Вирус гепатита В (HBV), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-010", "report_name": "Вирус гепатита C (HCV), генотипирование (типы 1a, 1b, 2, 3a, 4), РНК [реал-тайм ПЦР]"},
    {"hxid": "09-011", "report_name": "Вирус гепатита C (HCV), РНК [реал-тайм ПЦР]"},
    {"hxid": "09-012", "report_name": "Вирус гепатита C (HCV), РНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-013", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1/2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-015", "report_name": "Вирус герпеса человека 6-го типа (Human Herpes Virus 6), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-016", "report_name": "Вирус герпеса человека 7-го типа (Human Herpes Virus 7), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-022", "report_name": "Возбудитель туберкулеза (Mycobacterium tuberculosis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-023", "report_name": "Возбудители туберкулеза (Mycobacterium tuberculosis/bovis.), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-039", "report_name": "Возбудитель болезни Лайма (Borrelia burgdorferi s.l.), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-067", "report_name": "Возбудитель респираторного хламидиоза (Chlamydia pneumoniae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-068", "report_name": "Возбудитель токсоплазмоза (Toxoplasma gondii), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-071", "report_name": "Возбудитель микоплазменной пневмонии (Mycoplasma pneumoniae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-084", "report_name": "Вирус герпеса человека 8-го типа (Human Herpes Virus 8), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-123", "report_name": "Вирус ветряной оспы (Varicella Zoster Virus), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-151", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-152", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-164", "report_name": "Цитомегаловирус (Cytomegalovirus), ДНК, количественно [реал-тайм ПЦР]"},
    {"hxid": "09-168", "report_name": "Комплексное исследование на вирусы ЦМВ, ВЭБ, ВГЧ 6 (Cytomegalovirus, Epstein Barr Virus, Human Herpes Virus 6), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-181", "report_name": "Вирус Эпштейн-Барр (Epstein Barr Virus), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-182", "report_name": "Вирус герпеса человека 6-го типа (Human Herpes Virus 6), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "13-002", "report_name": "Аллоиммунные антиэритроцитарные антитела (в том числе антирезусные), титр (непрямая проба Кумбса)"},
    {"hxid": "13-003", "report_name": "Антитела к спермальным антигенам (в крови)"},
    {"hxid": "13-007", "report_name": "Антитела к двухцепочечной ДНК (анти-dsDNA), IgG"},
    {"hxid": "13-008", "report_name": "Антитела к инсулину, IgG"},
    {"hxid": "13-010", "report_name": "Антитела к рецепторам ТТГ (анти-pTTГ)"},
    {"hxid": "13-013", "report_name": "Антифосфолипидные антитела, IgM"},
    {"hxid": "13-014", "report_name": "Антитела к циклическому цитруллинсодержащему пептиду (АЦЦП), IgG"},
    {"hxid": "13-015", "report_name": "Антитела к ядерным антигенам (ANA), скрининг"},
    {"hxid": "13-016", "report_name": "Антитела к островковым клеткам поджелудочной железы, IgG"},
    {"hxid": "13-017", "report_name": "Антитела к глиадину, IgA"},
    {"hxid": "13-018", "report_name": "Антитела к глиадину, IgG"},
    {"hxid": "13-019", "report_name": "Антифосфолипидные антитела, IgG"},
    {"hxid": "13-020", "report_name": "Ревматоидный фактор"},
    {"hxid": "13-024", "report_name": "Антитела к бета-2-гликопротеину"},
    {"hxid": "13-025", "report_name": "Антикератиновые антитела (АКА), IgG"},
    {"hxid": "13-026", "report_name": "Антитела к цитруллинированному виментину (анти-MCV), IgG"},
    {"hxid": "13-028", "report_name": "Антитела к С1q фактору комплемента, IgG"},
    {"hxid": "13-030", "report_name": "Антитела к париетальным (обкладочным) клеткам желудка"},
    {"hxid": "13-031", "report_name": "Антитела к гладким мышцам"},
    {"hxid": "13-032", "report_name": "Антитела к эндомизию, IgA"},
    {"hxid": "13-033", "report_name": "Антитела к тканевой трансглутаминазе, IgG"},
    {"hxid": "13-034", "report_name": "Антитела к тканевой трансглутаминазе, IgA"},
    {"hxid": "13-041", "report_name": "Антитела к стероидпродуцирующим клеткам надпочечника"},
    {"hxid": "13-045", "report_name": "Антинуклеарный фактор на HEp-2-клетках, IgG"},
    {"hxid": "13-046", "report_name": "Антитела к экстрагируемому ядерному антигену (ENA-скрин)"},
    {"hxid": "13-047", "report_name": "Антитела к кардиолипину, IgG и IgM"},
    {"hxid": "13-048", "report_name": "Антиперинуклеарный фактор, IgG"},
    {"hxid": "13-050", "report_name": "Ангиотензинпревращающий фермент сыворотки"},
    {"hxid": "13-052", "report_name": "Антитела к цитоплазме нейтрофилов, IgG (с определением типа свечения)"},
    {"hxid": "13-053", "report_name": "Антитела к миелопероксидазе (anti-МРО, IgG)"},
    {"hxid": "13-054", "report_name": "Антитела к протеиназе-3 (anti-PR-3, IgG)"},
    {"hxid": "13-060", "report_name": "Диагностика системной красной волчанки"},
    {"hxid": "13-061", "report_name": "Диагностика антифосфолипидного синдрома (АФС)"},
    {"hxid": "13-062", "report_name": "Развернутая диагностика антифосфолипидного синдрома (АФС)"},
    {"hxid": "13-063", "report_name": "Антинуклеарные антитела, IgG (анти-Sm, RNP/Sm, SS-A, SS-B, Scl-70, PM-Scl, PCNA, dsDNA, CENT-B, Jo-1, к гистонам, к нуклеосомам, Ribo P, AMA-M2), иммуноблот"},
    {"hxid": "13-065", "report_name": "Диагностика гранулематозных васкулитов"},
    {"hxid": "13-068", "report_name": "Антитела к антигенам аутоиммунных заболеваний печени (растворимому антигену печени (SLA/LP), антитела к  цитозольному антигену (LC-1),антитела к микросомам печени-почек 1 типа (LKM-1), антитела к митохондриям М2 (AMA-M2), пируват-декарбоксилазному комплексу митохондрий (PDC-M23E), антитела к антигену Sp100, антитела к антигенам  PML и gp210), IgG (иммуноблот)"},
    {"hxid": "13-072", "report_name": "Дифференциальная диагностика болезни Крона и язвенного колита"},
    {"hxid": "13-077", "report_name": "Иммуноблот при полимиозите (Мi2b, Ku, Pm-Scl100, PM-Scl75, Jo-1, SRP, PL-7, PL-12 EJ, OJ, Ro-52)"},
    {"hxid": "13-078", "report_name": "Целиакия. Скрининг (взрослые и дети старше 2 лет)"},
    {"hxid": "13-079", "report_name": "Целиакия. Расширенное серологическое обследование"},
    {"hxid": "13-081", "report_name": "Панель антител цитоплазмы нейтрофилов (АНЦА)"},
    {"hxid": "13-083", "report_name": "Антитела к внутреннему фактору Кастла, IgG"},
    {"hxid": "13-087", "report_name": "Антитела к тиреоглобулину (антиТГ)"},
    {"hxid": "13-088", "report_name": "Антитела к тиреопероксидазе (антиТПО)"},
    {"hxid": "13-089", "report_name": "Антитела к глутаматдекарбоксилазе (анти-GAD), IgG"},
    {"hxid": "13-092", "report_name": "Диагностика миастении (антитела к ацетилхолиновому рецептору (АхР))"},
    {"hxid": "13-096", "report_name": "Диагностика аутоиммунного панкреатита (определение концентрации антител класса IgG4)"},
    {"hxid": "13-099", "report_name": "Скрининг миеломной болезни и парапротеинемий (иммунофиксация сыворотки крови с пентавалентной сывороткой)"},
    {"hxid": "13-102", "report_name": "Антитела к двуспиральной ДНК, IgG (высокоспецифичный метод)"},
    {"hxid": "13-106", "report_name": "Диагностика мембранозной нефропатии (антитела к рецептору фосфолипазы А2, IgG)"},
    {"hxid": "13-107", "report_name": "Антитела к сахаромицетам (Sacchаromyces cerevisiae), (ASCA, IgG)"},
    {"hxid": "13-108", "report_name": "Антитела к сахаромицетам (Sacchаromyces cerevisiae), (ASCA, IgA)"},
    {"hxid": "13-109", "report_name": "Антитела к цитоплазме нейтрофилов (АНЦА), IgA"},
    {"hxid": "13-110", "report_name": "Антитела к бокаловидным клеткам кишечника, IgG"},
    {"hxid": "13-119", "report_name": "Антитела к аннексину V, IgG"},
    {"hxid": "13-120", "report_name": "Антитела к аннексину V, IgM"},
    {"hxid": "13-121", "report_name": "Комбинированное обследование при воспалительных заболеваниях кишечника"},
    {"hxid": "13-124", "report_name": "Типирование парапротеина в сыворотке крови (с помощью иммунофиксации с панелью антисывороток IgG, IgA, IgM, IgD, IgE, kappa, lambda)"},
    {"hxid": "13-129", "report_name": "Определение активности ингибитора С1 фактора комплемента (C1INH)"},
    {"hxid": "13-130", "report_name": "Антитела к кардиолипину, IgG"},
    {"hxid": "13-131", "report_name": "Антитела к кардиолипину, IgМ"},
    {"hxid": "13-132", "report_name": "Антитела к бета-2-гликопротеину, IgМ"},
    {"hxid": "13-133", "report_name": "Антитела к дезаминированным пептидам глиадина, IgA"},
    {"hxid": "13-134", "report_name": "Антитела к дезаминированным пептидам глиадина, IgG"},
    {"hxid": "13-137", "report_name": "Определение общей гемолитической способности комплемента (CH-50)"},
    {"hxid": "13-144", "report_name": "Антигрупповые антитела со стандартными эритроцитами (естественные анти-А, анти-В, иммунные неполные анти-А, анти-В)"},
    {"hxid": "13-145", "report_name": "Фенотипирование эритроцитов по антигенам системы Rh (C, E, c, e) и Kell"},
    {"hxid": "13-146", "report_name": "Антитела к бета-2-гликопротеину, IgG"},
    {"hxid": "13-148", "report_name": "Антитела к мышечно-специфической тирозинкиназе (MUSK, IgG)"},
    {"hxid": "13-150", "report_name": "ЭЛИ-Висцеро-Тест-24 (антитела к 24 антигенам основных органов и систем человека)"},
    {"hxid": "13-152", "report_name": "ЭЛИ-АФС-ХГЧ Тест (антифосфолипидный синдром, анти-ХГЧ синдром, 6 антигенов)"},
    {"hxid": "13-155", "report_name": "Антитела к цитоплазматическому антигену SS-A(RO) (анти-Ro/SS-A)"},
    {"hxid": "13-157", "report_name": "Антитела к цитоплазматическому антигену SS-B(La) (Анти-La/SS-B)"},
    {"hxid": "13-158", "report_name": "Антитела к антигену Scl-70 (Анти-Scl-70)"},
    {"hxid": "13-162", "report_name": "Электрофорез липидов с расчетом триглицеридов"},
    {"hxid": "13-163", "report_name": "Электрофорез липидов с расчетом холестерина"},
    {"hxid": "13-165", "report_name": "Электрофорез гемоглобина для диагностики гемоглобинопатий"},
    {"hxid": "13-166", "report_name": "Антитела к NR2 субъединице NMDA рецептора глутамата в сыворотке"},
    {"hxid": "13-169", "report_name": "ЭЛИ-Н-ТЕСТ"},
    {"hxid": "15-001", "report_name": "Вальпроевая кислота"},
    {"hxid": "15-007", "report_name": "Ламотриджин"},
    {"hxid": "15-008", "report_name": "Дигоксин"},
    {"hxid": "15-011", "report_name": "Леветирацетам"},
    {"hxid": "15-028", "report_name": "Окскарбазепин"},
    {"hxid": "15-031", "report_name": "Такролимус"},
    {"hxid": "16-001", "report_name": "Исследование кариотипа (количественные и структурные аномалии хромосом) по лимфоцитам периферической крови (1 человек)"},
    {"hxid": "18-001", "report_name": "Ген рака молочной железы 1 (BRCA1). Выявление мутации 185delAG (нарушение структуры белка)"},
    {"hxid": "18-002", "report_name": "Ген рака молочной железы 1 (BRCA1). Выявление мутации 4153delA (нарушение структуры белка)"},
    {"hxid": "18-003", "report_name": "Ген рака молочной железы 1 (BRCA1). Выявление мутации 5382insC (нарушение структуры белка)"},
    {"hxid": "18-005", "report_name": "Ген рака молочной железы 2 (BRCA2). Выявление мутации 6174delT (нарушение структуры белка)"},
    {"hxid": "18-008", "report_name": "Метилентетрагидрофолатредуктаза (MTHFR). Выявление мутации A1298C (Glu429Ala)"},
    {"hxid": "18-009", "report_name": "Метионинсинтаза (MTR). Выявление мутации A2756G (Asp919Gly)"},
    {"hxid": "18-010", "report_name": "Метионин-синтаза-редуктаза (MTRR). Выявление мутации A66G (Ile22Met)"},
    {"hxid": "18-011", "report_name": "Ангиотензин-превращающий фермент (ACE). Выявление мутации Alu Ins/Del (регуляторная область гена)"},
    {"hxid": "18-013", "report_name": "Ген МСМ6. Исследование генетического маркера C(-13910)T (регуляторная область гена LAC)"},
    {"hxid": "18-016", "report_name": "Витамин К - редуктаза (VKORC1). Выявление мутации G(-1639)A (регуляторная область гена)"},
    {"hxid": "18-017", "report_name": "Рецептор дофамина D2 (DRD2). Выявление мутации C2137T (Glu713Lys)"},
    {"hxid": "18-019", "report_name": "Метилентетрагидрофолатредуктаза (MTHFR). Выявление мутации C677T (Ala222Val)"},
    {"hxid": "18-024", "report_name": "LOC727677 (LOC727677). Выявление мутации G(g.41686854)T (регуляторная область гена)"},
    {"hxid": "18-026", "report_name": "Фактор свертываемости крови 7 (F7). Выявление мутации G10976A (Arg353Gln)"},
    {"hxid": "18-030", "report_name": "Фактор свертываемости крови 5 (F5). Выявление мутации G1691A (Arg506Gln)"},
    {"hxid": "18-031", "report_name": "Фактор свертываемости крови 2, протромбин (F2). Выявление мутации G20210A (регуляторная область гена)"},
    {"hxid": "18-071", "report_name": "УДФ-глюкуронозил трансфераза 1A1 (UGT1A1). Выявление мутации (TA)6/7 (регуляторная область гена)"},
    {"hxid": "18-072", "report_name": "Аполипопротеин E (ApoE). Выявление полиморфизма e2-e3-e4"},
    {"hxid": "18-073", "report_name": "АМФ-дезаминаза (AMPD1). Выявление мутации C34T"},
    {"hxid": "18-086", "report_name": "Диагностика целиакии (типирование HLA DQ2/DQ8)"},
    {"hxid": "18-094", "report_name": "ПЦР-анализ мутации V617F в 14-м экзоне JAK2-гена"},
    {"hxid": "18-095", "report_name": "ПЦР-анализ мутаций в 12-м экзоне JAK2-гена"},
    {"hxid": "18-102", "report_name": "ПЦР-анализ мутаций в гене MPL"},
    {"hxid": "18-105", "report_name": "ПЦР-анализ мутаций, делеций, инсерций в гене CALR"},
    {"hxid": "18-135", "report_name": "Ген рецептора витамина D (VDR). Выявление мутации G283A (BsmI)"},
    {"hxid": "18-136", "report_name": "Типирование гена HLA-B51 для диагностики болезни Бехчета"},
    {"hxid": "18-139", "report_name": "Киназа контрольной точки 2 (CHEK2). Выявление мутации 1100delC и IVS2+1G>A"},
    {"hxid": "20-001", "report_name": "Фактор некроза опухоли-альфа (ФНО-альфа)"},
    {"hxid": "20-003", "report_name": "Интерлейкин-1-бета"},
    {"hxid": "20-005", "report_name": "Интерлейкин-6 в сыворотке"},
    {"hxid": "20-006", "report_name": "Интерлейкин-8"},
    {"hxid": "20-012", "report_name": "Свободные каппа- и лямбда-цепи иммуноглобулинов в сыворотке, IgG"},
    {"hxid": "20-019", "report_name": "С3 компонент комплемента"},
    {"hxid": "20-020", "report_name": "С4 компонент комплемента"},
    {"hxid": "20-024", "report_name": "Циркулирующие иммунные комплексы (ЦИК)"},
    {"hxid": "20-058", "report_name": "Интерлейкин-10"},
    {"hxid": "20-067", "report_name": "Иммунологическое обследование первичное"},
    {"hxid": "20-072", "report_name": "Определение антигена HLA-B27 с помощью метода проточной цитометрии"},
    {"hxid": "20-075", "report_name": "Расширенное иммунологическое обследование"},
    {"hxid": "20-077", "report_name": "Фенотипирование лимфоцитов (основные субпопуляции) - CD3, CD4, CD8, CD19, CD56"},
    {"hxid": "20-079", "report_name": "Иммунорегуляторный индекс, субпопуляции T-лимфоцитов"},
    {"hxid": "20-081", "report_name": "В-лимфоциты, % и абсолютное количество (CD19+ лимфоциты, B-cells, Percent and Absolute)"},
    {"hxid": "20-082", "report_name": "Субпопуляции В-лимфоцитов (CD19+CD5+, CD19+CD5-, CD19+CD5-CD27+)"},
    {"hxid": "21-022", "report_name": "Аллерген f92 - банан, IgE"},
    {"hxid": "21-036", "report_name": "Аллерген f216 - капуста кочанная, IgE"},
    {"hxid": "21-039", "report_name": "Аллерген f2 - коровье молоко, IgE"},
    {"hxid": "21-045", "report_name": "Аллерген f221 - кофе, IgE"},
    {"hxid": "21-056", "report_name": "Аллерген f31 - морковь, IgE"},
    {"hxid": "21-084", "report_name": "Аллерген f105 - шоколад, IgE"},
    {"hxid": "21-088", "report_name": "Аллерген f1 - яичный белок, IgE"},
    {"hxid": "21-092", "report_name": "Аллерген f233 - овомукоид, IgE"},
    {"hxid": "21-093", "report_name": "Аллерген f245 - яйцо куриное, IgE"},
    {"hxid": "21-095", "report_name": "Аллерген f161 - молочная сыворотка, IgE"},
    {"hxid": "21-100", "report_name": "Аллерген f82 - сыр \"моулд\", IgE"},
    {"hxid": "21-1012", "report_name": "Смесь аллергенов злаковых трав gx3 (ImmunoCAP), IgE: колосок душистый, плевел, тимофеевка луговая, рожь посевная, бухарник пушистый"},
    {"hxid": "21-1050", "report_name": "Аллергочип ALEX2 (до 300 аллергокомпонентов + IgE общий)"},
    {"hxid": "21-178", "report_name": "Аллерген k84 - масло подсолнечное, IgE"},
    {"hxid": "21-184", "report_name": "Аллерген e4 - перхоть коровы, IgE"},
    {"hxid": "21-186", "report_name": "Аллерген e87 - крыса, IgE"},
    {"hxid": "21-187", "report_name": "Аллерген e74 - моча крысы, IgE"},
    {"hxid": "21-225", "report_name": "Аллерген f256 - орех грецкий, IgE"},
    {"hxid": "21-289", "report_name": "Аллерген f11 - гречневая мука, IgG"},
    {"hxid": "21-303", "report_name": "Аллерген f2 - коровье молоко, IgG"},
    {"hxid": "21-305", "report_name": "Аллерген f221 - кофе, IgG"},
    {"hxid": "21-329", "report_name": "Аллерген f4 - пшеничная мука, IgG"},
    {"hxid": "21-337", "report_name": "Аллерген f25 - томаты, IgG"},
    {"hxid": "21-342", "report_name": "Аллерген f17 - фундук, IgG"},
    {"hxid": "21-346", "report_name": "Аллерген e2 - эпителий собаки, IgG"},
    {"hxid": "21-348", "report_name": "Аллерген f1 - яичный белок, IgG"},
    {"hxid": "21-350", "report_name": "Аллерген f232 - овальбумин, IgG"},
    {"hxid": "21-352", "report_name": "Аллерген f245 - яйцо куриное, IgG"},
    {"hxid": "21-353", "report_name": "Аллерген f231 - кипяченое молоко, IgG"},
    {"hxid": "21-357", "report_name": "Аллерген f78 - казеин, IgG"},
    {"hxid": "21-359", "report_name": "Аллерген f82 - сыр \"моулд\", IgG"},
    {"hxid": "21-371", "report_name": "Аллерген f79 - клейковина (глютен), IgG"},
    {"hxid": "21-377", "report_name": "Аллерген f9 - рис, IgG"},
    {"hxid": "21-391", "report_name": "Аллерген f90 - солод, IgG"},
    {"hxid": "21-396", "report_name": "Аллерген f291 - капуста цветная, IgG"},
    {"hxid": "21-406", "report_name": "Аллерген f89 - горчица, IgG"},
    {"hxid": "21-517", "report_name": "Аллерген c2 - пенициллин V, IgG"},
    {"hxid": "21-533", "report_name": "Смесь клещевых аллергенов № 1 (IgE): Dermatophagoides pteronyssinus, Dermatophagoides farinae, Dermatophagoides microceras, Acarus siro, Lepidoglyphus destructor, Tyrophagus putreus, Glycyphagus domesticus, Euroglyphus maynei"},
    {"hxid": "21-534", "report_name": "Смесь аллергенов деревьев № 1 (IgE): клён ясенелистый, берёза, дуб, вяз, маслина, грецкий орех"},
    {"hxid": "21-537", "report_name": "Смесь аллергенов деревьев № 5 (IgE): ольха, лещина обыкновенная, вяз, ива белая, тополь"},
    {"hxid": "21-538", "report_name": "Смесь аллергенов сорных трав № 1 (IgE): амброзия обыкновенная, полынь обыкновенная, подорожник, марь белая, поташник"},
    {"hxid": "21-541", "report_name": "Смесь пищевых аллергенов № 1 (IgE): арахис, миндаль, фундук, кокос, бразильский орех"},
    {"hxid": "21-543", "report_name": "Смесь пищевых аллергенов № 5 (IgE): яичный белок, коровье молоко, треска, пшеничная мука, арахис, соевые бобы"},
    {"hxid": "21-554", "report_name": "Смесь пищевых аллергенов № 51 (IgE): томаты, картофель, морковь, горох, фасоль белая"},
    {"hxid": "21-555", "report_name": "Смесь пищевых аллергенов № 73 (IgE): свинина, куриное мясо, говядина, мясо индейки"},
    {"hxid": "21-557", "report_name": "Смесь ингаляционных аллергенов № 2 (IgE): тимофеевка, Alternaria alternata (tenuis), берёза, полынь обыкновенная"},
    {"hxid": "21-558", "report_name": "Смесь ингаляционных аллергенов № 3 (IgE): Dermatophagoides pteronyssinus, эпителий кошки, эпителий собаки, Aspergillus fumigatus"},
    {"hxid": "21-559", "report_name": "Смесь ингаляционных аллергенов № 6 (IgE): Dermatophagoides pteronyssinus, Dermatophagoides farinae, перхоть собаки, перхоть кошки, тимофеевка луговая, Alternaria alternata (tenuis), берёза бородавчатая, полынь обыкновенная"},
    {"hxid": "21-561", "report_name": "Смесь бытовых аллергенов № 9 (IgE): Dermatophagoides farinae, эпителий кошки, перхоть лошади, перхоть собаки, Alternaria alternata (tenuis)"},
    {"hxid": "21-563", "report_name": "Смесь аллергенов плесени № 1 (IgG): Penicillium notatum, Aspergillus fumigatus, Alternaria tenuis, Cladosporium herbarum, Candida albicans"},
    {"hxid": "21-584", "report_name": "Смесь пищевых аллергенов № 2 (IgG): треска, тунец, креветки, лосось,  мидии"},
    {"hxid": "21-585", "report_name": "Смесь пищевых аллергенов № 5 (IgG): яичный белок, коровье молоко, треска, пшеничная мука, арахис, соевые бобы"},
    {"hxid": "21-600", "report_name": "Смесь ингаляционных аллергенов № 3 (IgG): Dermatophagoides pteronyssinus, эпителий кошки, эпителий собаки, Aspergillus fumigatus"},
    {"hxid": "21-604", "report_name": "Аллерген c68 - артикаин/ультракаин, IgE"},
    {"hxid": "21-605", "report_name": "Аллерген c88 - мепивакаин/полокаин, IgE"},
    {"hxid": "21-606", "report_name": "Аллерген c82 - лидокаин/ксилокаин, IgE"},
    {"hxid": "21-607", "report_name": "Аллерген c83 - прокаин/новокаин, IgE"},
    {"hxid": "21-608", "report_name": "Аллерген c86 - бензокаин, IgE"},
    {"hxid": "21-610", "report_name": "Аллерген c89 - бупивакаин/анекаин/маркаин, IgE"},
    {"hxid": "21-611", "report_name": "Аллерген c210 - тетракаин/дикаин, IgE"},
    {"hxid": "21-612", "report_name": "Аллерген k40 - никель, IgE"},
    {"hxid": "21-613", "report_name": "Аллерген k41 - хром, IgE"},
    {"hxid": "21-617", "report_name": "Аллерген k46 - кобальт, IgE"},
    {"hxid": "21-620", "report_name": "Аллерген e1 - эпителий и перхоть кошки, IgE (ImmunoCAP)"},
    {"hxid": "21-621", "report_name": "Аллерген e5 - перхоть собаки, IgE (ImmunoCAP)"},
    {"hxid": "21-622", "report_name": "Аллерген f245 - яйцо, IgE (ImmunoCAP)"},
    {"hxid": "21-623", "report_name": "Аллерген f83 - мясо курицы, IgE (ImmunoCAP)"},
    {"hxid": "21-624", "report_name": "Аллерген f1 - яичный белок, IgE (ImmunoCAP)"},
    {"hxid": "21-625", "report_name": "Аллерген f75 - яичный желток, IgE (ImmunoCAP)"},
    {"hxid": "21-627", "report_name": "Аллерген f2 - молоко коровье, IgE (ImmunoCAP)"},
    {"hxid": "21-628", "report_name": "Аллерген f27 - говядина, IgE (ImmunoCAP)"},
    {"hxid": "21-629", "report_name": "Аллерген f231 - кипяченое молоко, IgE (ImmunoCAP)"},
    {"hxid": "21-630", "report_name": "Аллергокомпонент f78 - казеин nBos d8, IgE (ImmunoCAP)"},
    {"hxid": "21-631", "report_name": "Аллерген d2 - клещ домашней пыли Dermatophagoides farinae, IgE (ImmunoCAP)"},
    {"hxid": "21-632", "report_name": "Аллерген h1 - домашняя пыль (Greer), IgE (ImmunoCAP)"},
    {"hxid": "21-633", "report_name": "Аллерген h2 - домашняя пыль (Hollister), IgE (ImmunoCAP)"},
    {"hxid": "21-636", "report_name": "Аллерген f79 - глютен (клейковина), IgE (ImmunoCAP)"},
    {"hxid": "21-637", "report_name": "Аллерген f5 - рожь, ржаная мука, IgE (ImmunoCAP)"},
    {"hxid": "21-640", "report_name": "Аллерген f41 - лосось, IgE (ImmunoCAP)"},
    {"hxid": "21-643", "report_name": "Аллерген f33 - апельсин, IgE (ImmunoCAP)"},
    {"hxid": "21-647", "report_name": "Аллерген f49 - яблоко, IgE (ImmunoCAP)"},
    {"hxid": "21-650", "report_name": "Аллерген f44 - клубника, IgE (ImmunoCAP)"},
    {"hxid": "21-657", "report_name": "Аллерген t3 - берёза бородавчатая, IgE (ImmunoCAP)"},
    {"hxid": "21-660", "report_name": "Аллерген t2 - ольха серая, IgE (ImmunoCAP)"},
    {"hxid": "21-661", "report_name": "Аллерген g6 - тимофеевка луговая, IgE (ImmunoCAP)"},
    {"hxid": "21-662", "report_name": "Смесь бытовых аллергенов hx2 (ImmunoCAP), IgE: домашняя пыль, клещ домашней пыли D. pteronyssinus, клещ домашней пыли D. farinae, таракан рыжий"},
    {"hxid": "21-663", "report_name": "Смесь аллергенов плесени mx1 (ImmunoCAP), IgE:  Penicillium chrysogenum, Cladosporium herbarum, Aspergillus fumigatus, Alternaria alternata"},
    {"hxid": "21-664", "report_name": "Смесь аллергенов злаковых трав gx1 (ImmunoCAP), IgE: ежа сборная, овсяница луговая, плевел, тимофеевка луговая, мятлик луговой"},
    {"hxid": "21-666", "report_name": "Смесь аллергенов животных ex2 (ImmunoCAP), IgE: перхоть кошки, перхоть собаки, эпителий морской свинки, крыса, мышь"},
    {"hxid": "21-667", "report_name": "Смесь аллергенов сорных трав wx5 (ImmunoCAP), IgE: амброзия высокая, полынь, нивяник, одуванчик, золотарник"},
    {"hxid": "21-668", "report_name": "Смесь пищевых аллергенов fx5 (ImmunoCAP), IgE: яичный белок, молоко, треска, пшеница, арахис, соя"},
    {"hxid": "21-670", "report_name": "Смесь аллергенов сорных трав wx3 (ImmunoCAP), IgE: полынь, подорожник ланцетовидный, марь, золотарник, крапива двудомная"},
    {"hxid": "21-674", "report_name": "Аллерген f4 - пшеница, пшеничная мука, IgE (ImmunoCAP)"},
    {"hxid": "21-675", "report_name": "Фадиатоп (ImmunoCAP)"},
    {"hxid": "21-676", "report_name": "Фадиатоп детский (ImmunoCAP)"},
    {"hxid": "21-677", "report_name": "Аллерген f14 - соя, IgE (ImmunoCAP)"},
    {"hxid": "21-678", "report_name": "Аллерген d1 - клещ домашней пыли Dermatophagoides pteronyssinus, IgE (ImmunoCAP)"},
    {"hxid": "21-681", "report_name": "Аллергокомпонент t215 - берёза rBet v1 PR-10, IgE (ImmunoCAP)"},
    {"hxid": "21-682", "report_name": "Аллергокомпонент f232 - овальбумин яйца nGal d2, IgE (ImmunoCAP)"},
    {"hxid": "21-683", "report_name": "Аллергокомпонент f233 - овомукоид яйца nGal d1, IgE (ImmunoCAP)"},
    {"hxid": "21-695", "report_name": "Аллерген f24 - креветки, IgE (ImmunoCAP)"},
    {"hxid": "21-698", "report_name": "Аллерген m6 -  плесневый гиб Alternaria alternata, IgE (ImmunoCAP)"},
    {"hxid": "21-700", "report_name": "Аллерген m2 - кладоспорий травяной (Cladosporium herbarum), IgE (ImmunoCAP)"},
    {"hxid": "21-702", "report_name": "Смесь аллергенов деревьев tx9 (ImmunoCAP), IgE: ольха серая, берёза бородавчатая, лещина, дуб, ива"},
    {"hxid": "21-703", "report_name": "Аллергокомпонент g213 - тимофеевка луговая  rPhl p1, rPhl p5b, IgE (ImmunoCAP)"},
    {"hxid": "21-712", "report_name": "Аллергокомпонент e204 - бычий сывороточный альбумин nBos d6, IgE (ImmunoCAP)"},
    {"hxid": "21-714", "report_name": "Аллерген t4 - лещина обыкновенная, IgE (ImmunoCAP)"},
    {"hxid": "21-729", "report_name": "Аллерген e6 - эпителий морской свинки, IgE (ImmunoCAP)"},
    {"hxid": "21-733", "report_name": "Аллерген f13 - арахис, IgE (ImmunoCAP)"},
    {"hxid": "21-738", "report_name": "Суммарные иммуноглобулины E (IgE) в сыворотке (ImmunoCAP)"},
    {"hxid": "21-739", "report_name": "Аллергокомпонент e94 - кошка rFel d1, IgE (ImmunoCAP)"},
    {"hxid": "21-741", "report_name": "Аллергокомпонент e101 - собака rCan f 1, IgE (ImmunoCAP)"},
    {"hxid": "21-750", "report_name": "Аллерген f79 - глютен, IgE, ИФА"},
    {"hxid": "21-753", "report_name": "Аллерген e2 - эпителий собаки, IgE, ИФА"},
    {"hxid": "21-754", "report_name": "Аллерген e78 - перо волнистого попугая, IgE, ИФА"},
    {"hxid": "21-757", "report_name": "Аллерген f105 - шоколад, IgE, ИФА"},
    {"hxid": "21-762", "report_name": "Аллерген f77 - бета-лактоглобулин, IgE, ИФА"},
    {"hxid": "21-763", "report_name": "Аллерген f78 - казеин, IgE, ИФА"},
    {"hxid": "21-768", "report_name": "Аллерген k20 - шерсть, IgE, ИФА"},
    {"hxid": "21-771", "report_name": "Аллерген p1 - аскарида (Ascaris lumbricoides), IgE, ИФА"},
    {"hxid": "21-776", "report_name": "Смесь аллергенов пыли hm1 (IgE): домашняя пыль, Dermatophagoides pteronyssinus, Dermatophagoides farinae, таракан-прусак, ИФА"},
    {"hxid": "21-780", "report_name": "Смесь бытовых аллергенов dm1 (IgE): Dermatophagoides pteronyssinus, Dermatophagoides farinae, эпителий кошки, эпителий собаки, ИФА"},
    {"hxid": "21-782", "report_name": "Аллерген e82 - Кролик, эпителий, IgE (ImmunoCAP)"},
    {"hxid": "21-784", "report_name": "Аллерген f12 - горох, IgE (ImmunoCAP)"},
    {"hxid": "21-788", "report_name": "Аллерген f218 - перец красный сладкий (паприка), IgE (ImmunoCAP)"},
    {"hxid": "21-791", "report_name": "Аллерген f258 - кальмар, IgE (ImmunoCAP)"},
    {"hxid": "21-796", "report_name": "Аллерген m1 - пеницилл золотистый (Penicillium notatum (P.chrysogenum)), IgE (ImmunoCAP)"},
    {"hxid": "21-807", "report_name": "Аллерген f20 - миндаль, IgE (ImmunoCAP)"},
    {"hxid": "21-813", "report_name": "Аллерген f222 - чай, IgE (ImmunoCAP)"},
    {"hxid": "21-819", "report_name": "Аллерген f256 - грецкий орех, IgE (ImmunoCAP)"},
    {"hxid": "21-822", "report_name": "Аллерген f302 - мандарин, IgE (ImmunoCAP)"},
    {"hxid": "21-827", "report_name": "Аллерген f40 - тунец, IgE (ImmunoCAP)"},
    {"hxid": "21-840", "report_name": "Аллерген m80 - стафилококковый энтеротоксин А, IgE (ImmunoCAP)"},
    {"hxid": "21-841", "report_name": "Аллерген m81 - стафилококковый энтеротоксин В, IgE (ImmunoCAP)"},
    {"hxid": "21-853", "report_name": "Аллерген c1 - пенициллин G, IgE (ImmunoCAP)"},
    {"hxid": "21-856", "report_name": "Аллерген e77 - помёт волнистого попугайчика, IgE (ImmunoCAP)"},
    {"hxid": "21-861", "report_name": "Аллерген f202 - орех кешью, IgE (ImmunoCAP)"},
    {"hxid": "21-870", "report_name": "Аллерген f236 - сыворотка коровьего молока, IgE (ImmunoCAP)"},
    {"hxid": "21-918", "report_name": "Аллерген k84 - семена подсолнечника, IgE (ImmunoCAP)"},
    {"hxid": "21-928", "report_name": "Смесь аллергенов перьев птиц ex71 (ImmunoCAP), IgE: гуся, курицы, утки, индейки"},
    {"hxid": "21-933", "report_name": "Смесь пищевых аллергенов fx22 (ImmunoCAP), IgE: орех пекан, кешью, фисташки, грецкий орех"},
    {"hxid": "21-960", "report_name": "Смесь пищевых аллергенов fx3 (ImmunoCAP), IgE: пшеница, овес, кукуруза, кунжутное семя, гречиха"},
    {"hxid": "21-961", "report_name": "Смесь пищевых аллергенов fx14 (ImmunoCAP), IgE: помидор, шпинат, капуста, красный перец"},
    {"hxid": "21-962", "report_name": "Смесь пищевых аллергенов fx15 (ImmunoCAP), IgE: апельсин, яблоко, банан, персик"},
    {"hxid": "21-988", "report_name": "Аллерген e73 - эпителий крысы, IgE (ImmunoCAP)"},
    {"hxid": "21-989", "report_name": "Аллерген e74 - протеины мочи крысы, IgE (ImmunoCAP)"},
    {"hxid": "21-997", "report_name": "Аллерген e211 - протеины мочи кролика, IgE (ImmunoCAP)"},
    {"hxid": "40-001", "report_name": "4 обязательных анализа, экспресс"},
    {"hxid": "40-003", "report_name": "Панель тестов на внутриутробные инфекции (TORCH-IgG)"},
    {"hxid": "40-008", "report_name": "Группа крови и резус-фактор"},
    {"hxid": "40-023", "report_name": "Лабораторная диагностика анемий"},
    {"hxid": "40-035", "report_name": "Панель тестов на внутриутробные инфекции (TORCH)"},
    {"hxid": "40-039", "report_name": "Липидограмма"},
    {"hxid": "40-046", "report_name": "Планирование беременности - гормональные анализы"},
    {"hxid": "40-063", "report_name": "Клинический и биохимический анализы крови - основные показатели"},
    {"hxid": "40-065", "report_name": "Развернутое лабораторное обследование щитовидной железы"},
    {"hxid": "40-080", "report_name": "Вирусные гепатиты. Первичная диагностика"},
    {"hxid": "40-082", "report_name": "Вирусный гепатит A. Обследование контактных лиц"},
    {"hxid": "40-083", "report_name": "Вирусный гепатит B. Анализы перед вакцинацией"},
    {"hxid": "40-087", "report_name": "Вирусный гепатит В. Обследование для исключения вируса гепатита В, в том числе у контактных лиц"},
    {"hxid": "40-089", "report_name": "Вирусный гепатит B. Эффективность проведенной вакцинации и определение необходимости ревакцинации"},
    {"hxid": "40-094", "report_name": "Вирусный гепатит C. Анализы для первичного выявления заболевания. Обследование контактных лиц"},
    {"hxid": "40-101", "report_name": "Нарушения менструального цикла (гормональный профиль)"},
    {"hxid": "40-105", "report_name": "Менопауза (гормональный профиль)"},
    {"hxid": "40-120", "report_name": "Билирубин и его фракции (общий, прямой и непрямой)"},
    {"hxid": "40-121", "report_name": "Баланс андрогенов"},
    {"hxid": "40-131", "report_name": "Лабораторная диагностика железодефицитной анемии"},
    {"hxid": "40-132", "report_name": "Развернутая лабораторная диагностика анемий"},
    {"hxid": "40-133", "report_name": "Лабораторное обследование при ревматоидном артрите"},
    {"hxid": "40-136", "report_name": "Общий лабораторный скрининг (онкологический)"},
    {"hxid": "40-137", "report_name": "Лабораторные маркеры рака молочной железы"},
    {"hxid": "40-141", "report_name": "Атероскрин оптимальный"},
    {"hxid": "40-148", "report_name": "Мужской гормональный статус - базовые лабораторные показатели"},
    {"hxid": "40-149", "report_name": "Женский гормональный статус - базовые лабораторные показатели"},
    {"hxid": "40-151", "report_name": "Оценка функции надпочечников"},
    {"hxid": "40-154", "report_name": "Лабораторные маркеры рака яичников"},
    {"hxid": "40-155", "report_name": "Лабораторные маркеры рака толстой кишки"},
    {"hxid": "40-156", "report_name": "Целиакия. Скрининг при селективном дефиците иммуноглобулинов IgA"},
    {"hxid": "40-157", "report_name": "Хеликобактер пилори (Helicobacter pylori), серологическая диагностика"},
    {"hxid": "40-162", "report_name": "Лабораторная диагностика и мониторинг атрофического гастрита и язвенной болезни"},
    {"hxid": "40-163", "report_name": "Серологическая диагностика кори, паротита и краснухи"},
    {"hxid": "40-168", "report_name": "Лабораторная диагностика инфекционного мононуклеоза"},
    {"hxid": "40-170", "report_name": "Лабораторная диагностика коклюша и паракоклюша"},
    {"hxid": "40-174", "report_name": "Серологическая диагностика клещевого боррелиоза и энцефалита"},
    {"hxid": "40-274", "report_name": "ФиброМакс"},
    {"hxid": "40-275", "report_name": "СтеатоСкрин"},
    {"hxid": "40-335", "report_name": "Онкопрофилактика женщин"},
    {"hxid": "40-372", "report_name": "Индекс здоровья простаты (PHI) - комплексная оценка риска рака предстательной железы"},
    {"hxid": "40-424", "report_name": "Комплексное исследование на гормоны (12 показателей)"},
    {"hxid": "40-439", "report_name": "Обследование щитовидной железы"},
    {"hxid": "40-440", "report_name": "Онкомаркеры для женщин"},
    {"hxid": "40-441", "report_name": "Лабораторная диагностика рака желудка"},
    {"hxid": "40-445", "report_name": "Компонентная диагностика аллергии на молоко"},
    {"hxid": "40-481", "report_name": "Первичное обследование щитовидной железы"},
    {"hxid": "40-482", "report_name": "Онкологический скрининг щитовидной железы"},
    {"hxid": "40-483", "report_name": "Лабораторное обследование функции печени"},
    {"hxid": "40-484", "report_name": "Развернутое лабораторное обследование печени"},
    {"hxid": "40-485", "report_name": "Развернутое лабораторное обследование поджелудочной железы"},
    {"hxid": "40-489", "report_name": "Развернутая диагностика сахарного диабета"},
    {"hxid": "40-490", "report_name": "Контроль компенсации сахарного диабета"},
    {"hxid": "40-492", "report_name": "Расширенное лабораторное обследование сердца и сосудов"},
    {"hxid": "40-498", "report_name": "Базовые биохимические показатели"},
    {"hxid": "40-500", "report_name": "Обследование печени: скрининг"},
    {"hxid": "40-501", "report_name": "Антитела к вирусу краснухи (Rubella, IgG) с определением авидности"},
    {"hxid": "40-502", "report_name": "Антитела к капсидному антигену вируса Эпштейна - Барр (VBA VCA, IgG) с определением авидности"},
    {"hxid": "40-503", "report_name": "Антитела к вирусу простого герпеса (HSV 1/2, IgG) с определением авидности"},
    {"hxid": "40-508", "report_name": "Метаболический баланс"},
    {"hxid": "40-535", "report_name": "Комплексное исследование на гормоны (6 показателей)"},
    {"hxid": "40-603", "report_name": "Спортивный. Перед началом занятий в тренажерном зале"},
    {"hxid": "40-604", "report_name": "Анализы для выбора тактики тренировок"},
    {"hxid": "40-607", "report_name": "Спортивный. Оценка баланса микроэлементов и витаминов"},
    {"hxid": "40-615", "report_name": "4 обязательных анализа"},
    {"hxid": "40-619", "report_name": "Оценка инсулинорезистентности (индекс HOMA-IR)"},
    {"hxid": "40-622", "report_name": "Биохимия для ФиброМакс"},
    {"hxid": "40-648", "report_name": "Анализ микробных маркеров методом газовой хромато-масс-спектрометрии по Осипову (зелено-желтый бланк)"},
    {"hxid": "40-660", "report_name": "Пепсиногены I и II с расчетом соотношения"},
    {"hxid": "40-682", "report_name": "Ринит/Астма (взрослый профиль)"},
    {"hxid": "40-688", "report_name": "Сахар под контролем: глюкоза + гликированный гемоглобин"},
    {"hxid": "40-689", "report_name": "Сила и тонус: витамин D + ферритин + железо"},
    {"hxid": "40-691", "report_name": "Расширенный скрининг для вегетарианцев и веганов"},
    {"hxid": "40-694", "report_name": "Гастроскрин"},
    {"hxid": "40-751", "report_name": "Липидограмма с расчетом сердечно-сосудистого риска по тропонину I"},
    {"hxid": "41-001", "report_name": "Кардиопрогноз"},
    {"hxid": "41-003", "report_name": "Скрининг функции щитовидной железы"},
    {"hxid": "41-004", "report_name": "Профилактика остеопороза"},
    {"hxid": "41-007", "report_name": "Онкопрофилактика для мужчин (ПСА общий + ПСА свободный)"},
    {"hxid": "41-009", "report_name": "Скрининг функции печени и поджелудочной железы"},
    {"hxid": "41-010", "report_name": "Первичная диагностика сахарного диабета"},
    {"hxid": "41-011", "report_name": "Первичная диагностика анемии"},
    {"hxid": "42-001", "report_name": "Предрасположенность к повышенной свертываемости крови"},
    {"hxid": "42-002", "report_name": "Предрасположенность к повышенному уровню гомоцистеина (гены ферментов фолатного цикла)"},
    {"hxid": "42-006", "report_name": "Биологический риск приёма гормональных контрацептивов"},
    {"hxid": "42-009", "report_name": "Генетический риск развития тромбофилии"},
    {"hxid": "42-012", "report_name": "Риск раннего развития рака молочной железы и яичников"},
    {"hxid": "42-018", "report_name": "Лактозная непереносимость (взрослые и дети старше 3 лет)"},
    {"hxid": "42-024", "report_name": "Наследственная гипербилирубинемия. Синдром Жильбера"},
    {"hxid": "42-053", "report_name": "Генодиагностика болезни Гентингтона. Ген HTT"},
    {"hxid": "42-070", "report_name": "Гемохроматоз 1 типа"},
    {"hxid": "42-087", "report_name": "Выявление гена HLA-B27. Определение предрасположенности к развитию  спондилоартропатий (в т.ч. анкилозирующего спондилита - болезнь Бехтерева)"},
    {"hxid": "42-094", "report_name": "Генодиагностика наследственной гиперхолестеринемии, комплексная. Гены APOB100, LDLR, PCSK9"},
    {"hxid": "42-100", "report_name": "Генотипирование HLA-Cw6 при псориазе"},
    {"hxid": "42-102", "report_name": "Определение резус-фактора плода (выявление гена RHD плода в крови матери)"},
    {"hxid": "42-127", "report_name": "Оценка влияния CYP2D6 и CYP2C19 на метаболизм антидепрессантов ингибиторов обратного захвата серотонина/норадреналина (эсциталопрам, циталопрам, сертралин, флувоксамин, пароксетин, венлафаксин)"},
    {"hxid": "42-128", "report_name": "Диагностика наследственной фруктоземии (p.A149P (c.448G>C ; p.Ala150Pro), p.A174D (.524C>A;p.Ala175Asp)  в гене ALDOB)"},
    {"hxid": "42-134", "report_name": "Комплексное исследование недостаточности протеина С, протеина S и антитромбина III при тромбофилии (экзоны 2, 7 гена SERPINC1, экзоны 11, 12 гена PROS1, экзоны 3, 7 гена PROC)"},
    {"hxid": "42-143", "report_name": "Генотипирование DPYD для оценки токсичности химиотерапии (DPYD*2A (c.1905+1G>A), DPYD *13 (c.1679T>G; p.Ile560Ser), c.2846A>T (p.Asp949Val) и HapB3 (c.1129–5923C>G)"},
    {"hxid": "42-144", "report_name": "Генодиагностика наследственного рака предстательной железы. Ген HOXB13 (патогенный вариант c.251G>A (p.Gly84Glu))"},
    {"hxid": "42-150", "report_name": "Расширенная диагностика лактазной недостаточности (MCM6 с.-13907 C>G, c.-13910 С>T, c.-13915 T>G, c.-14010 G>C)"},
    {"hxid": "80-064", "report_name": "Здоровье щитовидной железы: ТТГ, антиТПО"},
    {"hxid": "80-090", "report_name": "Бодрость и продуктивность: витамин D + ферритин"},
    {"hxid": "80-093", "report_name": "Базовый скрининг: витамин D + клинический анализ крови"},
    {"hxid": "80-100", "report_name": "Хорошее настроение: витамин D + ТТГ"},
    {"hxid": "80-110", "report_name": "Крепкие нервы и здоровый сон: витамин D + магний + железо"},
    {"hxid": "80-115", "report_name": "Общая оценка здоровья: базовая биохимия"},
    {"hxid": "80-123", "report_name": "Важные витамины D + B12 + B9"},
    {"hxid": "80-127", "report_name": "Здоровье от А до Я: витамин D + ТТГ+ общий анализ крови"},
    {"hxid": "80-128", "report_name": "Суперцена: креатинин + билирубин и его фракции + глюкоза"},
    {"hxid": "80-141", "report_name": "Гармония 360°"},
    {"hxid": "80-143", "report_name": "Все важное сразу"}
  ],
  "Соскоб с перианальных складок": [
    {"hxid": "02-053", "report_name": "Анализ на энтеробиоз"}
  ],
  "Мазок слизистой ротоглотки": [
    {"hxid": "09-003", "report_name": "Цитомегаловирус (Cytomegalovirus), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-006", "report_name": "Вирус Эпштейн-Барр (Epstein Barr Virus), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-013", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1/2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-015", "report_name": "Вирус герпеса человека 6-го типа (Human Herpes Virus 6), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-016", "report_name": "Вирус герпеса человека 7-го типа (Human Herpes Virus 7), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-065", "report_name": "Пиогенный стрептококк (Streptococcus pyogenes), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-067", "report_name": "Возбудитель респираторного хламидиоза (Chlamydia pneumoniae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-071", "report_name": "Возбудитель микоплазменной пневмонии (Mycoplasma pneumoniae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-151", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-152", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-164", "report_name": "Цитомегаловирус (Cytomegalovirus), ДНК, количественно [реал-тайм ПЦР]"},
    {"hxid": "09-168", "report_name": "Комплексное исследование на вирусы ЦМВ, ВЭБ, ВГЧ 6 (Cytomegalovirus, Epstein Barr Virus, Human Herpes Virus 6), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-181", "report_name": "Вирус Эпштейн-Барр (Epstein Barr Virus), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-182", "report_name": "Вирус герпеса человека 6-го типа (Human Herpes Virus 6), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "10-004", "report_name": "Посев на грибы (Candida spp.) с подбором антимикотических препаратов"},
    {"hxid": "10-009", "report_name": "Посев на гемолитический стрептококк группы А"},
    {"hxid": "10-010", "report_name": "Посев на дифтерийную палочку (Corynebacterium diphtheriae)"},
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "10-049", "report_name": "Посев на аэробную и факультативно-анаэробную флору"},
    {"hxid": "10-052", "report_name": "Посев на возбудителей коклюша и паракоклюша (Bordetella pertussis/parapertussis)"},
    {"hxid": "10-062", "report_name": "Посев на золотистый стафилококк (Staphylococcus aureus), количественный результат"},
    {"hxid": "10-067", "report_name": "Посев на золотистый стафилококк (Staphylococcus aureus), качественный результат"},
    {"hxid": "10-071", "report_name": "Посев на грибы родов Candida, Aspergillus, Cryptococcus с подбором антимикотических препаратов для кандиды (Candida spp.) (мазки различных локализаций)"},
    {"hxid": "18-013", "report_name": "Ген МСМ6. Исследование генетического маркера C(-13910)T (регуляторная область гена LAC)"},
    {"hxid": "40-164", "report_name": "Подтверждение инфицирования В-гемолитическим стрептококком группы А (St. Pyogenes)"},
    {"hxid": "42-001", "report_name": "Предрасположенность к повышенной свертываемости крови"},
    {"hxid": "42-018", "report_name": "Лактозная непереносимость (взрослые и дети старше 3 лет)"}
  ],
  "Кал": [
    {"hxid": "02-001", "report_name": "Анализ кала на скрытую кровь"},
    {"hxid": "02-009", "report_name": "Копрограмма"},
    {"hxid": "02-010", "report_name": "Анализ кала на яйца гельминтов"},
    {"hxid": "02-012", "report_name": "Анализ кала на цисты простейших"},
    {"hxid": "02-038", "report_name": "Скрытая кровь в кале, количественно (метод FOB Gold)"},
    {"hxid": "02-056", "report_name": "Анализ кала на яйца и личинки гельминтов, простейшие и их цисты (Parasep)"},
    {"hxid": "07-126", "report_name": "Выявление антигена лямблии (Giardia lamblia)"},
    {"hxid": "07-127", "report_name": "Выявление антигена хеликобактера пилори (Helicobacter pylori)"},
    {"hxid": "08-092", "report_name": "Копрологическая эластаза"},
    {"hxid": "09-053", "report_name": "Хеликобактер пилори (Helicobacter pylori), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-075", "report_name": "Энтеровирус (Enterovirus), РНК [реал-тайм ПЦР]"},
    {"hxid": "09-100", "report_name": "Сальмонеллы (Salmonella spp.), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-101", "report_name": "Возбудитель псевдотуберкулеза (Yersinia pseudotuberculosis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-102", "report_name": "Шигеллы (Shigella) и энтероинвазивные штаммы кишечной палочки (E. Coli), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-134", "report_name": "Острые кишечные инфекции, скрининг (Shigella spp., E. coli (EIEC), Salmonella spp., Campylobacter spp., Adenovirus F, Rotavirus A, Norovirus II, Astrovirus)"},
    {"hxid": "09-150", "report_name": "Возбудитель кишечного иерсиниоза (Yersinia enterocolitica), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-158", "report_name": "Норовирус (Norovirus II), РНК [реал-тайм ПЦР]"},
    {"hxid": "09-180", "report_name": "Острые кишечные инфекции (Rotavirus A, Norovirus II, Astrovirus)"},
    {"hxid": "10-013", "report_name": "Дисбактериоз кишечника с определением чувствительности к антибиотикам и бактериофагам"},
    {"hxid": "10-015", "report_name": "Посев кала на условно-патогенную флору с определением чувствительности к антибиотикам"},
    {"hxid": "10-039", "report_name": "Дисбактериоз кишечника с определением чувствительности к бактериофагам"},
    {"hxid": "10-040", "report_name": "Дисбактериоз кишечника без определения чувствительности к антибиотикам и бактериофагам"},
    {"hxid": "10-056", "report_name": "Посев кала на условно-патогенную флору без определения чувствительности к антибиотикам"},
    {"hxid": "10-057", "report_name": "Посев кала на патогенную флору (диз. группа и тифопаратифозная группа) без определения чувствительности к антибиотикам"},
    {"hxid": "10-062", "report_name": "Посев на золотистый стафилококк (Staphylococcus aureus), количественный результат"},
    {"hxid": "10-063", "report_name": "Дисбактериоз кишечника с определением антагонистической активности пробиотиков"},
    {"hxid": "10-064", "report_name": "Дисбактериоз кишечника с определением антагонистической активности пробиотиков и определением чувствительности к бактериофагам"},
    {"hxid": "10-065", "report_name": "Дисбактериоз кишечника"},
    {"hxid": "10-076", "report_name": "Посев на грибы (Candida spp.) с определением чувствительности к антимикотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "13-094", "report_name": "Кальпротектин в кале"},
    {"hxid": "13-127", "report_name": "Альфа-1-антитрипсин в кале, кишечная потеря белка"}
  ],
  "Отделяемое слизистой носа": [
    {"hxid": "02-033", "report_name": "Микроскопическое исследование мазка со слизистой оболочки носа"},
    {"hxid": "10-004", "report_name": "Посев на грибы (Candida spp.) с подбором антимикотических препаратов"},
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "10-049", "report_name": "Посев на аэробную и факультативно-анаэробную флору"},
    {"hxid": "10-062", "report_name": "Посев на золотистый стафилококк (Staphylococcus aureus), количественный результат"},
    {"hxid": "10-067", "report_name": "Посев на золотистый стафилококк (Staphylococcus aureus), качественный результат"}
  ],
  "Отделяемое мочеполовых органов": [
    {"hxid": "09-001", "report_name": "Возбудитель кандидоза (Candida albicans), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-002", "report_name": "Возбудитель хламидиоза (Chlamydia trachomatis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-007", "report_name": "Возбудитель гарднереллеза (Gardnerella vaginalis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-013", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1/2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-018", "report_name": "Вирус папилломы человека (Human Papillomavirus 16/18), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-020", "report_name": "Вирус папилломы человека (Human Papillomavirus 6/11), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-025", "report_name": "Возбудитель микоплазмоза (Mycoplasma genitalium), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-026", "report_name": "Возбудитель микоплазмоза (Mycoplasma hominis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-027", "report_name": "Гонококк (Neisseria gonorrhoeae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-030", "report_name": "Возбудитель трихомоноза (Trichomonas vaginalis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-031", "report_name": "Уреаплазма парвум (Ureaplasma parvum), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-032", "report_name": "Уреаплазма уреалитикум (Ureaplasma urealyticum), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-095", "report_name": "Уреаплазмы (Ureaplasma spp.), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-106", "report_name": "Вирус папилломы человека (Human Papillomavirus) высокого канцерогенного риска (16, 18, 31, 33, 35, 39, 45, 51, 52, 56, 58, 59), ДНК генотипирование [реал-тайм ПЦР]"},
    {"hxid": "09-113", "report_name": "Возбудитель сифилиса (Treponema pallidum), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-114", "report_name": "Уреаплазмы (Ureaplasma spp.), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-116", "report_name": "Фемофлор - 16 [реал-тайм ПЦР]"},
    {"hxid": "09-117", "report_name": "Фемофлор - 8 [реал- тайм ПЦР]"},
    {"hxid": "09-140", "report_name": "Вирус папилломы человека (Human Papillomavirus) низкого (6, 11, 44) и высокого (16, 18, 26, 31, 33, 35, 39, 45, 51, 52, 53, 56, 58, 59, 66, 68, 73, 82) канцерогенного риска, ДНК (выявление, генотипирование и количественное определение) [реал-тайм ПЦР] (Ро"},
    {"hxid": "09-151", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-152", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 2), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-155", "report_name": "Вирус папилломы человека (Human Papillomavirus 16/18), ДНК (выявление, генотипирование и количественное определение) [реал-тайм ПЦР]"},
    {"hxid": "09-159", "report_name": "Типирование и количественное определение ДНК грибов рода кандида (C. albicans, C. glabrata, C. krusei, C. parapsilosis / C. tropicalis)"},
    {"hxid": "09-163", "report_name": "Возбудитель микоплазмоза (Mycoplasma hominis), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-165", "report_name": "Диагностика бактериального вагиноза, ДНК количественно [реал-тайм ПЦР]"},
    {"hxid": "09-172", "report_name": "Развернутая диагностика ЗППП для мужчин (\"Андрофлор\"), ДНК количественно [реал-тайм ПЦР]"},
    {"hxid": "09-173", "report_name": "\"Андрофлор-скрин\", ДНК количественно [реал-тайм ПЦР]"},
    {"hxid": "09-175", "report_name": "Уреаплазма парвум (Ureaplasma parvum), ДНК [реал-тайм ПЦР],количественно"},
    {"hxid": "09-176", "report_name": "Уреаплазма уреалитикум (Ureaplasma urealyticum), ДНК [реал-тайм ПЦР], количественно"},
    {"hxid": "09-183", "report_name": "Вирус папилломы человека (Human Papillomavirus) высокого канцерогенного риска (16, 18, 31, 33, 35, 39, 45, 51, 52, 56, 58, 59, 66, 68 типы), ДНК, без определения типа [реал-тайм ПЦР]"},
    {"hxid": "09-184", "report_name": "Вирус папилломы человека (Human Papillomavirus) высокого канцерогенного риска (16, 18, 31, 33, 35, 39, 45, 51, 52, 56, 58, 59, 66, 68 типы), ДНК количественно, скрининг с определением возможности интеграции вируса в геном  [реал-тайм ПЦР]"},
    {"hxid": "09-187", "report_name": "Возбудитель микоплазмоза (Mycoplasma genitalium), ДНК, количественно [реал-тайм ПЦР]"},
    {"hxid": "09-188", "report_name": "Гонококк (Neisseria gonorrhoeae), ДНК [реал-тайм ПЦР],  количественно"},
    {"hxid": "09-190", "report_name": "Возбудитель хламидиоза (Chlamydia trachomatis), ДНК, количественно [реал-тайм ПЦР]"},
    {"hxid": "09-196", "report_name": "ФЛОРОСКРИН - комплексное исследование с выявлением гонококка, хламидий, микоплазм и трихомонад (NCMT)"},
    {"hxid": "40-042", "report_name": "Интимный - оптимальный - анализ мазка у женщин"},
    {"hxid": "40-043", "report_name": "Интимный - оптимальный - анализ мазка у мужчин"},
    {"hxid": "40-044", "report_name": "Планирование беременности - здоровье партнеров (для женщин)"},
    {"hxid": "40-118", "report_name": "Интимный - плюс - анализ мазка у женщин"},
    {"hxid": "40-119", "report_name": "Интимный - максимальный - анализ мазка у женщин"},
    {"hxid": "40-130", "report_name": "Интимный - плюс - для мужчин"},
    {"hxid": "40-499", "report_name": "Интимный - 9 тестов для мужчин"},
    {"hxid": "40-526", "report_name": "Фемофлор Скрин [реал-тайм ПЦР]"},
    {"hxid": "40-676", "report_name": "ИППП - ПЦР 6 инфекций"},
    {"hxid": "40-677", "report_name": "ИППП - ПЦР 9 инфекций"},
    {"hxid": "40-678", "report_name": "ИППП - ПЦР 10 инфекций"},
    {"hxid": "40-679", "report_name": "ИППП - ПЦР 12 инфекций"},
    {"hxid": "40-680", "report_name": "ИППП - ПЦР 13 инфекций"},
    {"hxid": "43-5245", "report_name": "Женские инфекции ЗППП"},
    {"hxid": "43-5246", "report_name": "ВПЧ-скрин для женщин"},
    {"hxid": "80-145", "report_name": "Женский профиль 360"}
  ],
  "Препарат цитологический": [
    {"hxid": "02-003", "report_name": "Микроскопическое исследование отделяемого мочеполовых органов женщин (микрофлора), 3 локализации"},
    {"hxid": "02-015", "report_name": "Микроскопическое исследование отделяемого мочеполовых органов мужчин (микрофлора)"},
    {"hxid": "02-057", "report_name": "Микроскопическое исследование отделяемого мочеполовых органов женщин (микрофлора), 2 локализации"},
    {"hxid": "40-042", "report_name": "Интимный - оптимальный - анализ мазка у женщин"},
    {"hxid": "40-043", "report_name": "Интимный - оптимальный - анализ мазка у мужчин"},
    {"hxid": "40-044", "report_name": "Планирование беременности - здоровье партнеров (для женщин)"},
    {"hxid": "40-119", "report_name": "Интимный - максимальный - анализ мазка у женщин"}
  ],
  "Кровь капиллярная": [
    {"hxid": "02-005", "report_name": "Клинический анализ крови (с лейкоцитарной формулой)"},
    {"hxid": "02-007", "report_name": "Скорость оседания эритроцитов (СОЭ)"},
    {"hxid": "02-014", "report_name": "Общий анализ крови (без лейкоцитарной формулы и СОЭ)"},
    {"hxid": "02-027", "report_name": "Ретикулоциты"},
    {"hxid": "02-029", "report_name": "Клинический анализ крови: общий анализ, лейкоцитарная формула, СОЭ (с микроскопией мазка крови при выявлении патологических изменений)"},
    {"hxid": "02-041", "report_name": "Клинический анализ крови с микроскопией лейкоцитарной формулы"},
    {"hxid": "02-043", "report_name": "Клинический анализ крови: общий анализ, лейкоцитарная формула, СОЭ (с обязательной микроскопией мазка крови)"},
    {"hxid": "06-003", "report_name": "Аланинаминотрансфераза (АЛТ)"},
    {"hxid": "06-010", "report_name": "Аспартатаминотрансфераза (АСТ)"},
    {"hxid": "06-015", "report_name": "Глюкоза в плазме"},
    {"hxid": "06-021", "report_name": "Креатинин в сыворотке (с определением СКФ)"},
    {"hxid": "06-034", "report_name": "Мочевина в сыворотке"},
    {"hxid": "06-036", "report_name": "Билирубин общий"},
    {"hxid": "40-120", "report_name": "Билирубин и его фракции (общий, прямой и непрямой)"}
  ],
  "Мазок ректальный": [
    {"hxid": "10-057", "report_name": "Посев кала на патогенную флору (диз. группа и тифопаратифозная группа) без определения чувствительности к антибиотикам"}
  ],
  "Мазок слизистой носоглотки": [
    {"hxid": "09-071", "report_name": "Возбудитель микоплазменной пневмонии (Mycoplasma pneumoniae), ДНК [реал-тайм ПЦР]"},
    {"hxid": "10-033", "report_name": "Вирус простого герпеса (Herpes Simplex Virus 1/2), ИФА"},
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "10-049", "report_name": "Посев на аэробную и факультативно-анаэробную флору"}
  ],
  "Мазок слизистой носоглотки и ротоглотки": [
    {"hxid": "09-038", "report_name": "Возбудитель коклюша (Bordetella pertussis), ДНК [реал-тайм ПЦР]"},
    {"hxid": "09-120", "report_name": "Вирусы гриппа А/В (Influenza virus A/B), РНК [реал-тайм ПЦР]"},
    {"hxid": "40-170", "report_name": "Лабораторная диагностика коклюша и паракоклюша"}
  ],
  "Соскоб с эндоцервикса": [
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"},
    {"hxid": "10-071", "report_name": "Посев на грибы родов Candida, Aspergillus, Cryptococcus с подбором антимикотических препаратов для кандиды (Candida spp.) (мазки различных локализаций)"}
  ],
  "Волос": [
    {"hxid": "06-197", "report_name": "Магний в волосах"},
    {"hxid": "06-209", "report_name": "Цинк в волосах"},
    {"hxid": "06-344", "report_name": "Олово в волосах"}
  ],
  "Отделяемое слизистой влагалища": [
    {"hxid": "10-004", "report_name": "Посев на грибы (Candida spp.) с подбором антимикотических препаратов"}
  ],
  "Отделяемое слизистой уретры": [
    {"hxid": "10-038", "report_name": "Посев на аэробную и факультативно-анаэробную флору с определением чувствительности к антибиотикам и подбором минимальной подавляющей концентрации препарата"}
  ]
}